In [1]:
# Day 6 — Additions to Weekly Report
import pandas as pd
import numpy as np
import yfinance as yf
import pickle
import warnings
warnings.filterwarnings('ignore')

# Load all existing data
with open('../data/price_data.pkl', 'rb') as f:
    price_data = pickle.load(f)

with open('../data/indicator_data.pkl', 'rb') as f:
    indicator_data = pickle.load(f)

fund_df    = pd.read_csv('../data/fundamental_scores.csv')
tech_df    = pd.read_csv('../data/technical_report.csv')

print(f"Price data     : {len(price_data)} stocks")
print(f"Indicator data : {len(indicator_data)} stocks")
print(f"Fundamental    : {len(fund_df)} stocks")
print(f"Tech report    : {len(tech_df)} stocks")
print(f"\nColumns in tech_df:")
print(tech_df.columns.tolist())

Price data     : 35 stocks
Indicator data : 35 stocks
Fundamental    : 35 stocks
Tech report    : 35 stocks

Columns in tech_df:
['Symbol', 'Sector', 'Fund Score', 'Current Price', 'RSI', 'ADX', 'MACD Hist', 'Vol Ratio', 'EMA20', 'EMA50', 'EMA200', 'DI+', 'DI-', 'Momentum Score', 'Reversal Score', 'Best Setup', 'Tech Score', 'Tier', 'In Consolidation', 'Consol Days', 'Consol Label', 'Pct to Breakout', 'Consol Volume', 'Sector Score', 'Cap Score', 'Market Cap Cr', 'Cap Category', 'Vol Label', 'Vol Inference', 'Sector Trend', 'Sector Detail', 'range_pct', 'Vol 5D Ratio', 'Breakout Vol']


In [2]:
# ── TASK 1: VOLUME INFERENCE ────────────────────────────────────────────────

def get_volume_inference(row):
    vol   = float(row['Vol Ratio'])
    setup = str(row['Best Setup'])
    rsi   = float(row['RSI'])
    adx   = float(row['ADX'])

    if   vol >= 2.0: vol_label = "Very High"
    elif vol >= 1.5: vol_label = "High"
    elif vol >= 1.0: vol_label = "Normal"
    elif vol >= 0.7: vol_label = "Low"
    else:            vol_label = "Very Low"

    # ── Momentum ──────────────────────────────────────────────────────────────
    if setup == 'Momentum':
        if   vol >= 2.0: return vol_label, "Strong participation → trend likely to continue"
        elif vol >= 1.5: return vol_label, "Good volume support → momentum confirmed"
        elif vol >= 1.0: return vol_label, "Average volume → monitor for increase"
        elif vol >= 0.7: return vol_label, "Weak volume → momentum may fade, wait"
        else:            return vol_label, "Very low volume → no conviction, caution"

    # ── Reversal ──────────────────────────────────────────────────────────────
    elif setup == 'Reversal':
        if   vol <= 0.5: return vol_label, "Sellers exhausted → reversal likely"
        elif vol <= 0.7: return vol_label, "Low volume on down move → selling pressure easing"
        elif vol <= 1.0: return vol_label, "Neutral volume → wait for low-volume base to form"
        elif vol <= 1.5: return vol_label, "Elevated volume on reversal → watch direction"
        else:            return vol_label, "High volume → could be capitulation or distribution"

    # ── Watching / Waiting ────────────────────────────────────────────────────
    else:
        if   adx > 40 and vol >= 1.5: return vol_label, "High volume downtrend → stay away"
        elif adx > 40:                return vol_label, "Strong downtrend → no entry signal"
        elif vol <= 0.5:              return vol_label, "Very low volume → no interest, wait"
        elif vol >= 1.5 and rsi < 35: return vol_label, "High volume selloff → possible capitulation"
        elif vol >= 1.5:              return vol_label, "Above avg volume but no setup → watch"
        else:                         return vol_label, "Low activity → no setup forming yet"


# ── Test ─────────────────────────────────────────────────────────────────────
print(f"{'Symbol':12} {'Setup':10} {'Vol':5}  {'Label':10}  Inference")
print("─" * 78)
for _, row in tech_df.iterrows():
    label, inference = get_volume_inference(row)
    print(f"  {row['Symbol']:12} {row['Best Setup']:10} "
          f"{row['Vol Ratio']:.2f}x  {label:10}  {inference}")

Symbol       Setup      Vol    Label       Inference
──────────────────────────────────────────────────────────────────────────────
  TCS          Watching   0.61x  Very Low    Strong downtrend → no entry signal
  RELIANCE     Watching   1.09x  Normal      Low activity → no setup forming yet
  WIPRO        Reversal   0.69x  Very Low    Low volume on down move → selling pressure easing
  PERSISTENT   Reversal   0.49x  Very Low    Sellers exhausted → reversal likely
  LTTS         Reversal   10.78x  Very High   High volume → could be capitulation or distribution
  HAPPSTMNDS   Momentum   0.72x  Low         Weak volume → momentum may fade, wait
  CHOLAFIN     Watching   0.85x  Low         Strong downtrend → no entry signal
  MUTHOOTFIN   Watching   1.17x  Normal      Strong downtrend → no entry signal
  MANAPPURAM   Watching   0.64x  Very Low    Strong downtrend → no entry signal
  CREDITACC    Reversal   0.73x  Low         Neutral volume → wait for low-volume base to form
  GARFIBRES    

In [3]:
# ── Add Vol Inference columns to tech_df ─────────────────────────────────────
tech_df['Vol Label']     = ''
tech_df['Vol Inference'] = ''

for idx, row in tech_df.iterrows():
    label, inference = get_volume_inference(row)
    tech_df.at[idx, 'Vol Label']     = label
    tech_df.at[idx, 'Vol Inference'] = inference

# ── Verify ────────────────────────────────────────────────────────────────────
print("Added to tech_df:")
print(tech_df[['Symbol', 'Best Setup', 'Vol Ratio',
               'Vol Label', 'Vol Inference']].to_string(index=False))

Added to tech_df:
    Symbol Best Setup  Vol Ratio Vol Label                                       Vol Inference
       TCS   Watching       0.61  Very Low                  Strong downtrend → no entry signal
  RELIANCE   Watching       1.09    Normal                 Low activity → no setup forming yet
     WIPRO   Reversal       0.69  Very Low   Low volume on down move → selling pressure easing
PERSISTENT   Reversal       0.49  Very Low                 Sellers exhausted → reversal likely
      LTTS   Reversal      10.78 Very High High volume → could be capitulation or distribution
HAPPSTMNDS   Momentum       0.72       Low               Weak volume → momentum may fade, wait
  CHOLAFIN   Watching       0.85       Low                  Strong downtrend → no entry signal
MUTHOOTFIN   Watching       1.17    Normal                  Strong downtrend → no entry signal
MANAPPURAM   Watching       0.64  Very Low                  Strong downtrend → no entry signal
 CREDITACC   Reversal       0.73

In [3]:
# ── COMPLETE SECTOR → NIFTY INDEX MAP ─────────────────────────────────────────
# Primary: NSE sectoral indices (^CNX...)
# Fallback: Nippon India ETFs (.NS) where index not available on yfinance

SECTOR_INDEX_MAP = {
    # ── Your 35 stocks sectors ─────────────────────────────────────────────────
    'Information Technology'             : '^CNXIT',
    'Healthcare'                         : '^CNXPHARMA',
    'Financial Services'                 : 'FINIETF.NS',    # ETF fallback (^CNXFIN failed)
    'Capital Goods'                      : '^CNXINFRA',
    'Chemicals'                          : 'CHEMIETF.NS',   # ETF fallback (^CNXCHEM failed)
    'Consumer Durables'                  : '^CNXCONSUM',
    'Oil, Gas & Consumable Fuels'        : '^CNXENERGY',
    'Automobile and Auto Components'     : '^CNXAUTO',
    'Textiles'                           : 'MID150BEES.NS', # ETF fallback (no index)
    'Consumer Services'                  : 'MID150BEES.NS', # ETF fallback

    # ── Additional sectors for Day 8 ──────────────────────────────────────────
    'Banking'                            : '^NSEBANK',
    'Private Banking'                    : 'PVTBANIETF.NS', # ETF fallback (^CNXPVTBANK failed)
    'PSU Banking'                        : '^CNXPSUBANK',
    'Fast Moving Consumer Goods'         : '^CNXFMCG',
    'Metals & Mining'                    : '^CNXMETAL',
    'Realty'                             : '^CNXREALTY',
    'Media Entertainment & Publication'  : '^CNXMEDIA',
    'Pharma'                             : '^CNXPHARMA',
    'Services'                           : '^CNXSERVICE',
    'Construction'                       : '^CNXINFRA',
    'Power'                              : '^CNXENERGY',
    'Defence'                            : 'MID150BEES.NS', # no dedicated index/ETF
    'Telecommunication'                  : 'MID150BEES.NS',
    'Diversified'                        : 'MID150BEES.NS',
    'Forest Materials'                   : 'MID150BEES.NS',
    'Agriculture'                        : 'MID150BEES.NS',
    'Cement'                             : '^CNXCMDT',
}

# ── Fetch data ─────────────────────────────────────────────────────────────────
print("Fetching sector index/ETF data...")
sector_price   = {}
ticker_data    = {}
unique_tickers = list(set(SECTOR_INDEX_MAP.values()))

for ticker in unique_tickers:
    try:
        df = yf.Ticker(ticker).history(period='1y')
        if len(df) > 50:
            ticker_data[ticker] = df
            print(f"  ✅ {ticker:20} {len(df)} days")
        else:
            print(f"  ⚠  {ticker:20} only {len(df)} days — FAILED")
    except Exception as e:
        print(f"  ❌ {ticker:20} ERROR: {e}")

# ── Map sector → price data ────────────────────────────────────────────────────
fallback = 'MID150BEES.NS'   # broad midcap ETF as last resort
for sector, ticker in SECTOR_INDEX_MAP.items():
    if ticker in ticker_data:
        sector_price[sector] = ticker_data[ticker]
    elif fallback in ticker_data:
        sector_price[sector] = ticker_data[fallback]
        print(f"  ⚠  {sector} → using MID150BEES fallback")

print(f"\nUnique tickers fetched : {len(ticker_data)}")
print(f"Sectors mapped         : {len(sector_price)}/{len(SECTOR_INDEX_MAP)}")

Fetching sector index/ETF data...
  ✅ ^CNXINFRA            247 days
  ✅ ^CNXMEDIA            247 days
  ✅ ^CNXPSUBANK          247 days
  ✅ ^CNXFMCG             247 days


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CHEMIETF.NS"}}}
$CHEMIETF.NS: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")


  ⚠  CHEMIETF.NS          only 0 days — FAILED
  ✅ ^CNXAUTO             247 days
  ✅ ^CNXMETAL            247 days
  ✅ ^CNXCMDT             247 days
  ✅ FINIETF.NS           248 days
  ✅ ^CNXSERVICE          247 days
  ✅ ^CNXIT               248 days
  ✅ ^NSEBANK             248 days
  ✅ ^CNXREALTY           247 days
  ✅ MID150BEES.NS        248 days
  ✅ ^CNXCONSUM           247 days
  ✅ PVTBANIETF.NS        248 days
  ✅ ^CNXENERGY           247 days
  ✅ ^CNXPHARMA           247 days
  ⚠  Chemicals → using MID150BEES fallback

Unique tickers fetched : 17
Sectors mapped         : 27/27


In [6]:
# ── Fix Chemicals — try Kotak Chemicals ETF ───────────────────────────────────
print("Testing Kotak Nifty Chemicals ETF...")
chem_df = yf.Ticker('KOTAKCHEMETF.NS').history(period='1y')
print(f"KOTAKCHEMETF.NS: {len(chem_df)} days")

if len(chem_df) > 50:
    ticker_data['KOTAKCHEMETF.NS'] = chem_df
    # Update Chemicals mapping
    for sector in ['Chemicals']:
        sector_price[sector] = chem_df
    print("✅ Chemicals → KOTAKCHEMETF.NS")
else:
    print("⚠  KOTAKCHEMETF.NS failed — Chemicals stays on MID150BEES fallback")
    print("   (acceptable — MID150BEES is broad midcap, chemicals are midcap heavy)")

# ── Final sector coverage summary ─────────────────────────────────────────────
print(f"\nFinal sector coverage: {len(sector_price)}/27")
for sector, df in sector_price.items():
    ticker = SECTOR_INDEX_MAP[sector]
    source = ticker if ticker in ticker_data else 'MID150BEES fallback'
    print(f"  {sector:40} {source}")

Testing Kotak Nifty Chemicals ETF...


$KOTAKCHEMETF.NS: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")


KOTAKCHEMETF.NS: 0 days
⚠  KOTAKCHEMETF.NS failed — Chemicals stays on MID150BEES fallback
   (acceptable — MID150BEES is broad midcap, chemicals are midcap heavy)

Final sector coverage: 27/27
  Information Technology                   ^CNXIT
  Healthcare                               ^CNXPHARMA
  Financial Services                       FINIETF.NS
  Capital Goods                            ^CNXINFRA
  Chemicals                                MID150BEES fallback
  Consumer Durables                        ^CNXCONSUM
  Oil, Gas & Consumable Fuels              ^CNXENERGY
  Automobile and Auto Components           ^CNXAUTO
  Textiles                                 MID150BEES.NS
  Consumer Services                        MID150BEES.NS
  Banking                                  ^NSEBANK
  Private Banking                          PVTBANIETF.NS
  PSU Banking                              ^CNXPSUBANK
  Fast Moving Consumer Goods               ^CNXFMCG
  Metals & Mining                         

In [4]:
# ── Calculate indicators on sector indices ────────────────────────────────────
def calculate_sector_indicators(df):
    data = df.copy()

    # EMA
    data['EMA20']  = data['Close'].ewm(span=20,  adjust=False).mean()
    data['EMA50']  = data['Close'].ewm(span=50,  adjust=False).mean()
    data['EMA200'] = data['Close'].ewm(span=200, adjust=False).mean()

    # RSI
    delta    = data['Close'].diff()
    gain     = delta.where(delta > 0, 0)
    loss     = -delta.where(delta < 0, 0)
    avg_gain = gain.ewm(span=14, adjust=False).mean()
    avg_loss = loss.ewm(span=14, adjust=False).mean()
    rs       = avg_gain / (avg_loss + 1e-9)
    data['RSI'] = 100 - (100 / (1 + rs))

    # MACD
    ema12          = data['Close'].ewm(span=12, adjust=False).mean()
    ema26          = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD']   = ema12 - ema26
    data['Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()
    data['MACD_Hist'] = data['MACD'] - data['Signal']

    # ADX
    high     = data['High']
    low      = data['Low']
    close    = data['Close']
    tr1      = high - low
    tr2      = (high - close.shift()).abs()
    tr3      = (low  - close.shift()).abs()
    tr       = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    dm_plus  = high.diff()
    dm_minus = -low.diff()
    dm_plus  = dm_plus.where((dm_plus > dm_minus) & (dm_plus > 0), 0)
    dm_minus = dm_minus.where((dm_minus > dm_plus) & (dm_minus > 0), 0)
    atr      = tr.ewm(span=14, adjust=False).mean()
    di_plus  = 100 * dm_plus.ewm(span=14, adjust=False).mean() / (atr + 1e-9)
    di_minus = 100 * dm_minus.ewm(span=14, adjust=False).mean() / (atr + 1e-9)
    dx       = 100 * (di_plus - di_minus).abs() / (di_plus + di_minus + 1e-9)
    data['ADX']      = dx.ewm(span=14, adjust=False).mean()
    data['DI_Plus']  = di_plus
    data['DI_Minus'] = di_minus

    return data


# ── Get sector trend label ─────────────────────────────────────────────────────
def get_sector_trend(sector):
    if sector not in sector_price:
        return "No Data", "N/A | N/A | N/A"

    df     = calculate_sector_indicators(sector_price[sector])
    latest = df.iloc[-1]

    close    = latest['Close']
    rsi      = round(latest['RSI'],      1)
    adx      = round(latest['ADX'],      1)
    ema20    = latest['EMA20']
    ema50    = latest['EMA50']
    ema200   = latest['EMA200']
    di_plus  = latest['DI_Plus']
    di_minus = latest['DI_Minus']
    macd_h   = latest['MACD_Hist']

    above_ema200 = close > ema200
    above_ema50  = close > ema50
    uptrend      = di_plus > di_minus
    strong       = adx > 25

    # Trend label
    if above_ema200 and above_ema50 and strong and uptrend and rsi > 55:
        label = "Strong Uptrend ↑"
    elif above_ema200 and uptrend and rsi > 50:
        label = "Uptrend ↑"
    elif above_ema200 and not strong:
        label = "Weak Uptrend →"
    elif not above_ema200 and not strong and rsi > 45:
        label = "Sideways →"
    elif not above_ema200 and strong and not uptrend and rsi < 45:
        label = "Downtrend ↓"
    elif not above_ema200 and strong and not uptrend and rsi < 35:
        label = "Strong Downtrend ↓"
    else:
        label = "Weak Downtrend ↓"

    ema_status = "Above EMA200" if above_ema200 else "Below EMA200"
    detail     = f"RSI {rsi} | ADX {adx} | {ema_status}"

    return label, detail


# ── Test all sectors in your 35 stocks ────────────────────────────────────────
sectors_in_use = tech_df['Sector'].unique()
print(f"{'Sector':40} {'Trend':20} Detail")
print("─" * 85)
for sector in sorted(sectors_in_use):
    label, detail = get_sector_trend(sector)
    print(f"  {sector:40} {label:20} {detail}")

Sector                                   Trend                Detail
─────────────────────────────────────────────────────────────────────────────────────
  Automobile and Auto Components           Downtrend ↓          RSI 37.2 | ADX 40.3 | Below EMA200
  Capital Goods                            Downtrend ↓          RSI 36.0 | ADX 40.7 | Below EMA200
  Chemicals                                Downtrend ↓          RSI 34.4 | ADX 42.3 | Below EMA200
  Consumer Durables                        Downtrend ↓          RSI 36.4 | ADX 46.3 | Below EMA200
  Consumer Services                        Downtrend ↓          RSI 34.4 | ADX 42.3 | Below EMA200
  Financial Services                       Downtrend ↓          RSI 32.5 | ADX 28.2 | Below EMA200
  Healthcare                               Weak Downtrend ↓     RSI 45.0 | ADX 27.7 | Above EMA200
  Information Technology                   Downtrend ↓          RSI 40.4 | ADX 54.5 | Below EMA200
  Oil, Gas & Consumable Fuels              Weak Uptre

In [5]:
# ── Add Sector Trend columns to tech_df ───────────────────────────────────────
tech_df['Sector Trend']  = ''
tech_df['Sector Detail'] = ''

for idx, row in tech_df.iterrows():
    label, detail = get_sector_trend(row['Sector'])
    tech_df.at[idx, 'Sector Trend']  = label
    tech_df.at[idx, 'Sector Detail'] = detail

# ── Verify ────────────────────────────────────────────────────────────────────
print(f"{'Symbol':12} {'Sector':35} {'Trend':20} Detail")
print("─" * 90)
for _, row in tech_df.iterrows():
    print(f"  {row['Symbol']:12} {row['Sector']:35} "
          f"{row['Sector Trend']:20} {row['Sector Detail']}")

Symbol       Sector                              Trend                Detail
──────────────────────────────────────────────────────────────────────────────────────────
  TCS          Information Technology              Downtrend ↓          RSI 40.4 | ADX 54.5 | Below EMA200
  RELIANCE     Oil, Gas & Consumable Fuels         Weak Uptrend →       RSI 46.6 | ADX 16.4 | Above EMA200
  WIPRO        Information Technology              Downtrend ↓          RSI 40.4 | ADX 54.5 | Below EMA200
  PERSISTENT   Information Technology              Downtrend ↓          RSI 40.4 | ADX 54.5 | Below EMA200
  LTTS         Information Technology              Downtrend ↓          RSI 40.4 | ADX 54.5 | Below EMA200
  HAPPSTMNDS   Information Technology              Downtrend ↓          RSI 40.4 | ADX 54.5 | Below EMA200
  CHOLAFIN     Financial Services                  Downtrend ↓          RSI 32.5 | ADX 28.2 | Below EMA200
  MUTHOOTFIN   Financial Services                  Downtrend ↓          RSI 32.5 | 

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPLETE WEEKLY REPORT — WITH VOLUME INFERENCE + SECTOR TREND
# ══════════════════════════════════════════════════════════════════════════════

from datetime import datetime

def classify_mcap(mcap_cr):
    if   mcap_cr >= 20000: return 'Large Cap'
    elif mcap_cr >= 10000: return 'Mini Large Cap'
    elif mcap_cr >= 2000:  return 'Mid Cap'
    else:                  return 'Small Cap'

def get_reason(row):
    rsi    = float(row['RSI'])
    adx    = float(row['ADX'])
    setup  = str(row['Best Setup'])
    hist   = float(row['MACD Hist'])
    di_p   = float(row.get('DI+', 0))
    di_m   = float(row.get('DI-', 0))
    ema50  = float(row.get('EMA50',  0))
    ema200 = float(row.get('EMA200', 0))
    price  = float(row.get('Current Price', 0))

    uptrend   = (adx > 25 and di_p > di_m and
                 price > ema50 and price > ema200)
    bounce    = (price > ema50 and price < ema200)
    downtrend = (price < ema200) or (adx > 25 and di_m > di_p)

    if setup == 'Momentum':
        if uptrend:
            return f"Uptrend confirmed, RSI {rsi:.0f}, ADX {adx:.0f}"
        elif bounce:
            return f"Short bounce, below EMA200"
        elif price < ema200 and rsi > 50:
            return f"RSI {rsi:.0f} bounce in downtrend"
        elif rsi >= 50:
            return f"Momentum building, RSI {rsi:.0f}"
        return f"Momentum setup, RSI {rsi:.0f}"

    elif setup == 'Reversal':
        if rsi < 35 and hist > 0:
            return f"Oversold RSI {rsi:.0f}, MACD turning up"
        elif rsi < 35:
            return f"Oversold RSI {rsi:.0f}, watch for turn"
        elif hist > 0 and downtrend:
            return f"Downtrend weakening, MACD improving"
        return f"Divergence forming, RSI {rsi:.0f}"

    else:
        if downtrend and adx > 40:
            return f"Strong downtrend ADX {adx:.0f}, wait"
        elif downtrend and adx > 25:
            return f"Downtrend ADX {adx:.0f}, no setup yet"
        elif bounce:
            return f"Bounce in downtrend, below EMA200"
        elif rsi < 25:
            return f"Deeply oversold RSI {rsi:.0f}, too early"
        elif rsi < 40:
            return f"Oversold RSI {rsi:.0f}, watch for turn"
        elif rsi > 60 and not uptrend:
            return f"RSI {rsi:.0f} bounce, below EMA200"
        else:
            return f"RSI {rsi:.0f}, setup not confirmed"


# ── Header ────────────────────────────────────────────────────────────────────
today = datetime.now().strftime('%d %B %Y')
print(f"{'='*78}")
print(f"  AI STOCK SCREENER — WEEKLY REPORT")
print(f"  {today}")
print(f"{'='*78}")
print(f"\n  SecRank = rank vs same sector  |  CapRank = rank vs same sector + same cap")

# ── TIER 1 ────────────────────────────────────────────────────────────────────
tier1 = tech_df[tech_df['Tier'].str.startswith('TIER 1')].copy()
tier1 = tier1.sort_values(['Cap Category', 'Fund Score'], ascending=[True, False])

cap_order = ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']

print(f"\n{'─'*78}")
print(f"  TIER 1 — ACT NOW  ({len(tier1)} stocks)")
print(f"  Fundamentally strong + Technical setup confirmed")
print(f"{'─'*78}")

current_cap = None
for _, row in tier1.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        print(f"\n  [{cap[:1]}{'L' if cap == 'Mini Large Cap' else ''}] {cap}")
        print(f"  {'─'*68}")

    consol_line = ''
    if row['In Consolidation'] and row['Consol Days'] > 0:
        consol_line = f"\n    Base    : {row['Consol Days']}d ({row['Consol Label']}) | {row['Pct to Breakout']:+.1f}% to breakout"

    reason    = get_reason(row)
    vol_label = row['Vol Label']
    vol_inf   = row['Vol Inference']
    sec_trend = row['Sector Trend']
    sec_det   = row['Sector Detail']

    print(f"""
    {row['Symbol']}  ({row['Sector']})  MCap Rs{row['Market Cap Cr']:,.0f}Cr
    Setup   : {row['Best Setup']:10}  Tech {row['Tech Score']:3.0f}/100
    Scores  : Fund {row['Fund Score']}/100  Sector Rank {row['Sector Score']}/10  Cap Rank {row['Cap Score']}/10
    Price   : Rs{row['Current Price']:.2f}  RSI {row['RSI']:.0f}  ADX {row['ADX']:.0f}  MACD {row['MACD Hist']:+.2f}{consol_line}
    Tech    : {reason}
    Volume  : {row['Vol Ratio']:.1f}x ({vol_label}) → {vol_inf}
    Sector  : {sec_trend} | {sec_det}""")

# ── TIER 2A ───────────────────────────────────────────────────────────────────
tier2a = tech_df[tech_df['Tier'] == 'TIER 2 — WATCHLIST'].copy()
tier2a = tier2a.sort_values('Fund Score', ascending=False)

print(f"\n{'─'*78}")
print(f"  TIER 2A — WATCHLIST  ({len(tier2a)} stocks)")
print(f"  Setup forming — wait for confirmation before entering")
print(f"{'─'*78}")
print(f"\n  {'Symbol':10} {'Fund':5} {'SecRnk':7} {'CapRnk':7} {'Setup':10} "
      f"{'Tech':4} {'RSI':4} {'ADX':4}  {'Vol':5}")
print(f"  {'Tech Reason'}")
print(f"  {'Vol Inference'}")
print(f"  {'Sector Trend'}")
print(f"  {'─'*76}")

for _, row in tier2a.iterrows():
    consol_tag = f" [{row['Consol Days']}d]" if row['In Consolidation'] else ''
    reason     = get_reason(row)
    vol_label  = row['Vol Label']
    vol_inf    = row['Vol Inference']
    sec_trend  = row['Sector Trend']
    sec_det    = row['Sector Detail']

    print(f"\n  {row['Symbol']:10} {row['Fund Score']:4.1f}  "
          f"{row['Sector Score']:5.1f}/10  {row['Cap Score']:5.1f}/10  "
          f"{row['Best Setup']:10} {row['Tech Score']:3.0f}  "
          f"{row['RSI']:4.0f}  {row['ADX']:4.0f}  "
          f"{row['Vol Ratio']:.1f}x{consol_tag}")
    print(f"    Tech  : {reason}")
    print(f"    Vol   : {vol_label} → {vol_inf}")
    print(f"    Sector: {sec_trend} | {sec_det}")

# ── TIER 2B ───────────────────────────────────────────────────────────────────
tier2b = tech_df[tech_df['Tier'] == 'TIER 2 — BASE BUILDING'].copy()
tier2b = tier2b.sort_values('Fund Score', ascending=False)

print(f"\n{'─'*78}")
print(f"  TIER 2B — BASE BUILDING  ({len(tier2b)} stocks)")
print(f"  Long consolidation base — watch for volume breakout")
print(f"{'─'*78}")
print(f"\n  {'Symbol':10} {'Fund':5} {'SecRnk':7} {'CapRnk':7} "
      f"{'Days':6} {'Range%':7} {'ToBreak':8} {'Vol':5}")
print(f"  {'Tech Reason'}")
print(f"  {'Vol Inference'}")
print(f"  {'Sector Trend'}")
print(f"  {'─'*76}")

for _, row in tier2b.iterrows():
    reason    = get_reason(row)
    vol_label = row['Vol Label']
    vol_inf   = row['Vol Inference']
    sec_trend = row['Sector Trend']
    sec_det   = row['Sector Detail']

    print(f"\n  {row['Symbol']:10} {row['Fund Score']:4.1f}  "
          f"{row['Sector Score']:5.1f}/10  {row['Cap Score']:5.1f}/10  "
          f"{row['Consol Days']:5}d  "
          f"{row.get('Range Pct', 0):6.1f}%  "
          f"{row['Pct to Breakout']:+7.1f}%  "
          f"{row['Consol Volume']:.1f}x")
    print(f"    Tech  : {reason}")
    print(f"    Vol   : {vol_label} → {vol_inf}")
    print(f"    Sector: {sec_trend} | {sec_det}")

# ── TIER 3 ────────────────────────────────────────────────────────────────────
tier3 = tech_df[tech_df['Tier'] == 'TIER 3 — WAITING'].copy()
tier3 = tier3.sort_values('Fund Score', ascending=False)

print(f"\n{'─'*78}")
print(f"  TIER 3 — WAITING  ({len(tier3)} stocks)")
print(f"  Good businesses — technical setup not ready yet")
print(f"{'─'*78}")

current_cap = None
for _, row in tier3.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        print(f"\n  [{cap[0]}{'L' if cap == 'Mini Large Cap' else ''}] {cap}")
        print(f"  {'Symbol':10} {'Fund':5} {'SecRnk':7} {'CapRnk':7} "
              f"{'RSI':4} {'ADX':4}  {'Vol':5}")
        print(f"  {'─'*70}")

    consol_tag = f" [{row['Consol Days']}d base]" if row['In Consolidation'] else ''
    reason     = get_reason(row)
    vol_inf    = row['Vol Inference']
    sec_trend  = row['Sector Trend']
    sec_det    = row['Sector Detail']

    print(f"\n  {row['Symbol']:10} {row['Fund Score']:4.1f}  "
          f"{row['Sector Score']:5.1f}/10  {row['Cap Score']:5.1f}/10  "
          f"{row['RSI']:4.0f}  {row['ADX']:4.0f}  "
          f"{row['Vol Ratio']:.1f}x{consol_tag}")
    print(f"    Tech  : {reason}")
    print(f"    Vol   : {row['Vol Label']} → {vol_inf}")
    print(f"    Sector: {sec_trend} | {sec_det}")

# ── LONG BASE WATCH ───────────────────────────────────────────────────────────
consol_stocks = tech_df[tech_df['In Consolidation'] == True].copy()
consol_stocks = consol_stocks.sort_values('Consol Days', ascending=False)

print(f"\n{'─'*78}")
print(f"  LONG BASE WATCH  ({len(consol_stocks)} stocks, 300+ days)")
print(f"  Watch for breakout above range high with 2x+ volume")
print(f"{'─'*78}")
print(f"  {'Symbol':12} {'Category':14} {'Fund':5} "
      f"{'SecRnk':7} {'CapRnk':7} "
      f"{'Days':6} {'Range%':7} {'ToBreak':8} {'Vol':5}  Status")
print(f"  {'─'*78}")

for _, row in consol_stocks.iterrows():
    pct = row['Pct to Breakout']
    if   pct > -5:   status = "NEAR BREAKOUT"
    elif pct > -15:  status = "Approaching"
    else:            status = "Early stage"

    range_pct = row.get('Range Pct', 0)

    print(f"  {row['Symbol']:12} "
          f"{row['Cap Category']:14} "
          f"{row['Fund Score']:4.1f}  "
          f"{row['Sector Score']:5.1f}/10  "
          f"{row['Cap Score']:5.1f}/10  "
          f"{row['Consol Days']:5}d  "
          f"{range_pct:6.1f}%  "
          f"{pct:+7.1f}%  "
          f"{row['Consol Volume']:.1f}x  "
          f"{status}")

print(f"\n{'─'*78}")
print(f"  L=Large Cap  ML=Mini Large Cap  M=Mid Cap  S=Small Cap")
print(f"  SecRank = rank vs same sector | CapRank = rank vs same sector + cap")
print(f"{'─'*78}")

# ── Save updated tech_df ──────────────────────────────────────────────────────
tech_df.to_csv('../data/technical_report.csv', index=False)
print("\nSaved: data/technical_report.csv ✅")

  AI STOCK SCREENER — WEEKLY REPORT
  16 March 2026

  SecRank = rank vs same sector  |  CapRank = rank vs same sector + same cap

──────────────────────────────────────────────────────────────────────────────
  TIER 1 — ACT NOW  (4 stocks)
  Fundamentally strong + Technical setup confirmed
──────────────────────────────────────────────────────────────────────────────

  [L] Large Cap
  ────────────────────────────────────────────────────────────────────

    SUNPHARMA  (Healthcare)  MCap Rs432,264Cr
    Setup   : Momentum    Tech 100/100
    Scores  : Fund 65.0/100  Sector Rank 10.0/10  Cap Rank 10.0/10
    Price   : Rs1801.60  RSI 61  ADX 40  MACD +5.11
    Base    : 520d (520d (1.5-2 years)) | -2.7% to breakout
    Tech    : Uptrend confirmed, RSI 61, ADX 40
    Volume  : 1.1x (Normal) → Average volume → monitor for increase
    Sector  : Weak Uptrend → | RSI 37.4 | ADX 22.1 | Above EMA200

  [ML] Mini Large Cap
  ────────────────────────────────────────────────────────────────────


In [6]:
# Check 1: Range Pct column
print("Columns in tech_df:", [c for c in tech_df.columns if 'Range' in c or 'Consol' in c])

# Check 2: Cap category sort order
print("\nTier 3 cap categories:")
tier3_check = tech_df[tech_df['Tier'] == 'TIER 3 — WAITING'].copy()
print(tier3_check[['Symbol', 'Cap Category', 'Fund Score']].sort_values(
    ['Cap Category', 'Fund Score'], ascending=[True, False]).to_string())

Columns in tech_df: ['In Consolidation', 'Consol Days', 'Consol Label', 'Consol Volume']

Tier 3 cap categories:
        Symbol    Cap Category  Fund Score
7   MUTHOOTFIN       Large Cap        73.4
6     CHOLAFIN       Large Cap        70.7
26       TITAN       Large Cap        69.0
14         BEL       Large Cap        68.0
0          TCS       Large Cap        64.0
22    DIVISLAB       Large Cap        64.0
1     RELIANCE       Large Cap        55.0
2        WIPRO       Large Cap        49.0
16       PARAS         Mid Cap        63.2
20       AAVAS         Mid Cap        60.8
17  VINATIORGA         Mid Cap        58.0
19  GALAXYSURF         Mid Cap        50.0
28      VGUARD         Mid Cap        49.0
29    SYMPHONY         Mid Cap        48.7
11    SUPRAJIT         Mid Cap        45.0
9    CREDITACC         Mid Cap        41.4
23      APLLTD         Mid Cap        39.0
15     MIDHANI         Mid Cap        35.0
8   MANAPPURAM         Mid Cap        31.9
31       PIIND  Mini Large 

In [7]:
# Fix 1: Load Range Pct from consolidation_info.csv
consol_info_df = pd.read_csv('../data/consolidation_info.csv')
print("Consolidation info columns:", consol_info_df.columns.tolist())
print(consol_info_df[['Symbol'] + [c for c in consol_info_df.columns if 'range' in c.lower() or 'Range' in c]].head(10))

Consolidation info columns: ['Symbol', 'Fund Score', 'Sector', 'consol_days', 'duration_label', 'range_pct', 'range_high', 'range_low', 'pct_to_breakout', 'is_breaking_out', 'breakout_volume', 'has_bell', 'current_price']
       Symbol  range_pct  range_high  range_low
0     FINEORG      37.43     5468.25    3978.82
1    RELIANCE      34.85     1559.75    1156.63
2   SUNPHARMA      26.06     1851.62    1468.86
3    SUPRAJIT      38.99      532.59     383.17
4         HAL      49.79     5142.18    3432.95
5     IPCALAB      37.83     1613.74    1170.79
6    GRANULES      47.87      667.07     451.11
7  PERSISTENT      36.38     6453.55    4732.07


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPLETE WEEKLY REPORT — CLEAN FORMAT
# ══════════════════════════════════════════════════════════════════════════════
from datetime import datetime

today = datetime.now().strftime('%d %B %Y')
print(f"{'='*78}")
print(f"  AI STOCK SCREENER — WEEKLY REPORT")
print(f"  {today}")
print(f"{'='*78}")
print(f"\n  SecRank = rank vs same sector  |  CapRank = rank vs same sector + same cap")

# ── TIER 1 — detailed card (unchanged) ────────────────────────────────────────
tier1 = tech_df[tech_df['Tier'].str.startswith('TIER 1')].copy()
tier1['_cap_order'] = tier1['Cap Category'].apply(lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier1 = tier1.sort_values(['_cap_order', 'Fund Score'], ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 1 — ACT NOW  ({len(tier1)} stocks)")
print(f"  Fundamentally strong + Technical setup confirmed")
print(f"{'─'*78}")

serial = 1
current_cap = None
for _, row in tier1.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML','Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_line = ''
    if row['In Consolidation'] and row['Consol Days'] > 0:
        consol_line = (f"\n    Base    : {row['Consol Days']}d "
                      f"({row['Consol Label']}) | "
                      f"{row['Pct to Breakout']:+.1f}% to breakout")

    print(f"""
  #{serial}  {row['Symbol']}  ({row['Sector']})  MCap Rs{row['Market Cap Cr']:,.0f}Cr
    Setup   : {row['Best Setup']:10}  Tech {row['Tech Score']:3.0f}/100
    Scores  : Fund {row['Fund Score']}/100  Sector Rank {row['Sector Score']}/10  Cap Rank {row['Cap Score']}/10
    Price   : Rs{row['Current Price']:.2f}  RSI {row['RSI']:.0f}  ADX {row['ADX']:.0f}  MACD {row['MACD Hist']:+.2f}{consol_line}
    Tech    : {get_reason(row)}
    Volume  : {row['Vol Ratio']:.1f}x ({row['Vol Label']}) → {row['Vol Inference']}
    Sector  : {row['Sector Trend']} | {row['Sector Detail']}""")
    serial += 1

# ── TIER 2A ────────────────────────────────────────────────────────────────────
tier2a = tech_df[tech_df['Tier'] == 'TIER 2 — WATCHLIST'].copy()
tier2a = tier2a.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2A — WATCHLIST  ({len(tier2a)} stocks)")
print(f"  Setup forming — wait for confirmation before entering")
print(f"{'─'*78}")

for _, row in tier2a.iterrows():
    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''
    # Line 1: identity only
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
    # Line 2: Technical — includes RSI, ADX, Setup, Tech score + reason
    print(f"      Technical : Setup:{row['Best Setup']:10}  Tech:{row['Tech Score']:3.0f}  "
          f"RSI:{row['RSI']:4.0f}  ADX:{row['ADX']:4.0f}  → {get_reason(row)}")
    # Line 3: Volume
    print(f"      Volume    : {row['Vol Ratio']:.1f}x {row['Vol Label']} → {row['Vol Inference']}")
    # Line 4: Sector
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── TIER 2B ────────────────────────────────────────────────────────────────────
tier2b = tech_df[tech_df['Tier'] == 'TIER 2 — BASE BUILDING'].copy()
tier2b = tier2b.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2B — BASE BUILDING  ({len(tier2b)} stocks)")
print(f"  Long consolidation base — watch for volume breakout")
print(f"{'─'*78}")

for _, row in tier2b.iterrows():
    # Line 1: identity + consolidation info only
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}  "
          f"{row['Consol Days']}d  "
          f"Range:{row['range_pct']:.1f}%  "
          f"ToBreak:{row['Pct to Breakout']:+.1f}%")
    # Line 2: Technical — RSI, ADX + reason
    print(f"      Technical : RSI:{row['RSI']:4.0f}  ADX:{row['ADX']:4.0f}  → {get_reason(row)}")
    # Line 3: Volume — both daily vol ratio AND consolidation breakout volume
    print(f"      Volume    : Daily:{row['Vol Ratio']:.1f}x {row['Vol Label']} → {row['Vol Inference']}  |  Breakout vol:{row['Consol Volume']:.1f}x")
    # Line 4: Sector
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── TIER 3 ─────────────────────────────────────────────────────────────────────
tier3 = tech_df[tech_df['Tier'] == 'TIER 3 — WAITING'].copy()
tier3['_cap_order'] = tier3['Cap Category'].apply(lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier3 = tier3.sort_values(['_cap_order', 'Fund Score'], ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 3 — WAITING  ({len(tier3)} stocks)")
print(f"  Good businesses — technical setup not ready yet")
print(f"{'─'*78}")

current_cap = None
for _, row in tier3.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML','Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''
    # Line 1: identity + Vol only
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}  "
          f"Vol:{row['Vol Ratio']:.1f}x{consol_tag}")
    # Line 2: Technical — RSI, ADX + reason
    print(f"      Technical : RSI:{row['RSI']:4.0f}  ADX:{row['ADX']:4.0f}  → {get_reason(row)}")
    # Line 3: Volume
    print(f"      Volume    : {row['Vol Label']} → {row['Vol Inference']}")
    # Line 4: Sector
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── LONG BASE WATCH ────────────────────────────────────────────────────────────
consol_stocks = tech_df[tech_df['In Consolidation'] == True].copy()
consol_stocks = consol_stocks.sort_values('Consol Days', ascending=False)

print(f"\n{'─'*78}")
print(f"  LONG BASE WATCH  ({len(consol_stocks)} stocks, 300+ days)")
print(f"  Watch for breakout above range high with 2x+ volume")
print(f"{'─'*78}")
print(f"  {'Symbol':12} {'Category':14} {'Fund':5} "
      f"{'Sec':6} {'Cap':6} "
      f"{'Days':6} {'Range%':7} {'ToBreak':8} {'Vol':5}  Status")
print(f"  {'─'*85}")

for _, row in consol_stocks.iterrows():
    pct    = row['Pct to Breakout']
    status = "NEAR BREAKOUT" if pct > -5 else "Approaching" if pct > -15 else "Early stage"
    print(f"  {row['Symbol']:12} "
          f"{row['Cap Category']:14} "
          f"{row['Fund Score']:4.1f} "
          f"{row['Sector Score']:5.1f} "
          f"{row['Cap Score']:5.1f} "
          f"{row['Consol Days']:5}d "
          f"{row['range_pct']:6.1f}% "
          f"{pct:+7.1f}% "
          f"{row['Consol Volume']:.1f}x  "
          f"{status}")

print(f"\n{'─'*78}")
print(f"  L=Large Cap  ML=Mini Large Cap  M=Mid Cap  S=Small Cap")
print(f"  SecRank = rank vs same sector | CapRank = rank vs same sector + cap")
print(f"{'─'*78}")

tech_df.to_csv('../data/technical_report.csv', index=False)
print("\nSaved: data/technical_report.csv ✅")

  AI STOCK SCREENER — WEEKLY REPORT
  21 March 2026

  SecRank = rank vs same sector  |  CapRank = rank vs same sector + same cap


NameError: name 'CAP_ORDER' is not defined

In [9]:
# ── Fix 1: Calculate 5-day avg volume ratio from price_data ───────────────────
print("Calculating 5-day average volume ratios...")

vol_5d_data = {}
for symbol, df in price_data.items():
    try:
        vol_5d_avg  = df['Volume'].iloc[-5:].mean()   # last 5 days avg
        vol_20d_avg = df['Volume'].iloc[-20:].mean()  # last 20 days avg
        vol_5d_ratio = vol_5d_avg / vol_20d_avg if vol_20d_avg > 0 else 1.0
        vol_5d_data[symbol] = round(vol_5d_ratio, 2)
    except:
        vol_5d_data[symbol] = 1.0

# Add to tech_df
tech_df['Vol 5D Ratio'] = tech_df['Symbol'].map(vol_5d_data)

# Verify — compare daily vs 5d for key stocks
print(f"\n{'Symbol':12} {'Daily Vol (today)':20} {'5D Avg Vol':20}  Difference")
print("─" * 70)
for _, row in tech_df.iterrows():
    daily = row['Vol Ratio']
    fived = row['Vol 5D Ratio']
    diff  = fived - daily
    flag  = " ← spike" if abs(diff) > 0.5 else ""
    print(f"  {row['Symbol']:12} {daily:6.2f}x (today)          "
          f"{fived:6.2f}x (5d avg)        {diff:+.2f}{flag}")

Calculating 5-day average volume ratios...

Symbol       Daily Vol (today)    5D Avg Vol            Difference
──────────────────────────────────────────────────────────────────────
  TCS            0.61x (today)            0.78x (5d avg)        +0.17
  RELIANCE       1.09x (today)            1.30x (5d avg)        +0.21
  WIPRO          0.69x (today)            1.09x (5d avg)        +0.40
  PERSISTENT     0.49x (today)            0.62x (5d avg)        +0.13
  LTTS          10.78x (today)            2.35x (5d avg)        -8.43 ← spike
  HAPPSTMNDS     0.72x (today)            3.69x (5d avg)        +2.97 ← spike
  CHOLAFIN       0.85x (today)            0.95x (5d avg)        +0.10
  MUTHOOTFIN     1.17x (today)            0.87x (5d avg)        -0.30
  MANAPPURAM     0.64x (today)            0.91x (5d avg)        +0.27
  CREDITACC      0.73x (today)            0.76x (5d avg)        +0.03
  GARFIBRES      3.65x (today)            1.51x (5d avg)        -2.14 ← spike
  SUPRAJIT       0.78x (

In [18]:
# ── Fix: use non-overlapping baseline ─────────────────────────────────────────
print("Calculating 5-day volume ratio vs prior 15-day baseline...")

vol_5d_data = {}
for symbol, df in price_data.items():
    try:
        if len(df) < 20:
            vol_5d_data[symbol] = 1.0
            continue

        vol_5d_avg      = df['Volume'].iloc[-5:].mean()    # last 5 days
        vol_prior15_avg = df['Volume'].iloc[-20:-5].mean() # prior 15 days (no overlap)

        vol_5d_ratio = vol_5d_avg / vol_prior15_avg if vol_prior15_avg > 0 else 1.0
        vol_5d_data[symbol] = round(vol_5d_ratio, 2)
    except:
        vol_5d_data[symbol] = 1.0

# Update tech_df
tech_df['Vol 5D Ratio'] = tech_df['Symbol'].map(vol_5d_data)

# Compare old vs new
print(f"\n{'Symbol':12} {'Old(vs20D)':12} {'New(vs15D)':12}  Change")
print("─" * 55)
for _, row in tech_df.iterrows():
    old = row['Vol Ratio']        # original daily vs 20D
    new = row['Vol 5D Ratio']     # 5D vs prior 15D
    flag = " ← different" if abs(new - old) > 0.3 else ""
    print(f"  {row['Symbol']:12} {old:6.2f}x       {new:6.2f}x      {flag}")

Calculating 5-day volume ratio vs prior 15-day baseline...

Symbol       Old(vs20D)   New(vs15D)    Change
───────────────────────────────────────────────────────
  TCS            0.61x         0.73x      
  RELIANCE       1.09x         1.44x       ← different
  WIPRO          0.69x         1.13x       ← different
  PERSISTENT     0.49x         0.55x      
  LTTS          10.78x         4.29x       ← different
  HAPPSTMNDS     0.72x        35.76x       ← different
  CHOLAFIN       0.85x         0.94x      
  MUTHOOTFIN     1.17x         0.83x       ← different
  MANAPPURAM     0.64x         0.88x      
  CREDITACC      0.73x         0.70x      
  GARFIBRES      3.65x         1.83x       ← different
  SUPRAJIT       0.78x         0.67x      
  HIMATSEIDE     1.39x         1.93x       ← different
  HAL            1.12x         0.79x       ← different
  BEL            1.05x         0.97x      
  MIDHANI        1.22x         1.02x      
  PARAS          0.28x         0.59x       ← differen

In [10]:
# ── Rebuild inferences with new 5D vs 15D ratio ───────────────────────────────
for idx, row in tech_df.iterrows():
    label, inference = get_volume_inference(row)
    tech_df.at[idx, 'Vol Label']     = label
    tech_df.at[idx, 'Vol Inference'] = inference

# ── Final verify ──────────────────────────────────────────────────────────────
print(f"{'Symbol':12} {'5D/15D':8} {'Label':12}  Inference")
print("─" * 85)
for _, row in tech_df.iterrows():
    print(f"  {row['Symbol']:12} {row['Vol 5D Ratio']:5.2f}x  "
          f"{row['Vol Label']:12}  {row['Vol Inference']}")

Symbol       5D/15D   Label         Inference
─────────────────────────────────────────────────────────────────────────────────────
  TCS           0.78x  Very Low      Strong downtrend → no entry signal
  RELIANCE      1.30x  Normal        Low activity → no setup forming yet
  WIPRO         1.09x  Very Low      Low volume on down move → selling pressure easing
  PERSISTENT    0.62x  Very Low      Sellers exhausted → reversal likely
  LTTS          2.35x  Very High     High volume → could be capitulation or distribution
  HAPPSTMNDS    3.69x  Low           Weak volume → momentum may fade, wait
  CHOLAFIN      0.95x  Low           Strong downtrend → no entry signal
  MUTHOOTFIN    0.87x  Normal        Strong downtrend → no entry signal
  MANAPPURAM    0.91x  Very Low      Strong downtrend → no entry signal
  CREDITACC     0.76x  Low           Neutral volume → wait for low-volume base to form
  GARFIBRES     1.51x  Very High     High volume → could be capitulation or distribution
  SUPRA

In [11]:
for idx, row in tech_df.iterrows():
    label, inference = get_volume_inference(row)
    tech_df.at[idx, 'Vol Label']     = label
    tech_df.at[idx, 'Vol Inference'] = inference

print(f"{'Symbol':12} {'5D/15D':8} {'Label':12}  Inference")
print("─" * 85)
for _, row in tech_df.iterrows():
    print(f"  {row['Symbol']:12} {row['Vol 5D Ratio']:5.2f}x  "
          f"{row['Vol Label']:12}  {row['Vol Inference']}")

Symbol       5D/15D   Label         Inference
─────────────────────────────────────────────────────────────────────────────────────
  TCS           0.78x  Very Low      Strong downtrend → no entry signal
  RELIANCE      1.30x  Normal        Low activity → no setup forming yet
  WIPRO         1.09x  Very Low      Low volume on down move → selling pressure easing
  PERSISTENT    0.62x  Very Low      Sellers exhausted → reversal likely
  LTTS          2.35x  Very High     High volume → could be capitulation or distribution
  HAPPSTMNDS    3.69x  Low           Weak volume → momentum may fade, wait
  CHOLAFIN      0.95x  Low           Strong downtrend → no entry signal
  MUTHOOTFIN    0.87x  Normal        Strong downtrend → no entry signal
  MANAPPURAM    0.91x  Very Low      Strong downtrend → no entry signal
  CREDITACC     0.76x  Low           Neutral volume → wait for low-volume base to form
  GARFIBRES     1.51x  Very High     High volume → could be capitulation or distribution
  SUPRA

In [12]:
def get_volume_inference(row):
    vol   = float(row['Vol 5D Ratio'])  # single source of truth
    setup = str(row['Best Setup'])
    rsi   = float(row['RSI'])
    adx   = float(row['ADX'])

    # ── Label based on 5D ratio ───────────────────────────────────────────────
    if   vol >= 3.0:  vol_label = "Extremely High"
    elif vol >= 2.0:  vol_label = "Very High"
    elif vol >= 1.5:  vol_label = "High"
    elif vol >= 1.0:  vol_label = "Normal"
    elif vol >= 0.7:  vol_label = "Low"
    else:             vol_label = "Very Low"

    # ── Momentum ──────────────────────────────────────────────────────────────
    if setup == 'Momentum':
        if   vol >= 3.0: return vol_label, "Exceptional participation → strong conviction"
        elif vol >= 2.0: return vol_label, "Strong participation → trend likely to continue"
        elif vol >= 1.5: return vol_label, "Good volume support → momentum confirmed"
        elif vol >= 1.0: return vol_label, "Average volume → monitor for increase"
        elif vol >= 0.7: return vol_label, "Weak volume → momentum may fade, wait"
        else:            return vol_label, "Very low volume → no conviction, caution"

    # ── Reversal ──────────────────────────────────────────────────────────────
    elif setup == 'Reversal':
        if   vol >= 3.0: return vol_label, "Very high volume on reversal → strong capitulation signal"
        elif vol >= 2.0: return vol_label, "High volume → could be capitulation or distribution"
        elif vol >= 1.5: return vol_label, "Elevated volume on reversal → watch direction carefully"
        elif vol >= 1.0: return vol_label, "Neutral volume → wait for low-volume base to form"
        elif vol >= 0.7: return vol_label, "Selling pressure easing → watch for turn"
        else:            return vol_label, "Sellers exhausted → reversal likely"

    # ── Watching ──────────────────────────────────────────────────────────────
    else:
        if   vol >= 3.0 and adx > 40:  return vol_label, "Very high volume downtrend → capitulation possible"
        elif vol >= 3.0:               return vol_label, "Exceptional volume → major event this week, watch"
        elif vol >= 2.0 and adx > 40:  return vol_label, "High volume downtrend → stay away"
        elif vol >= 2.0 and rsi < 35:  return vol_label, "High volume selloff → possible capitulation"
        elif vol >= 2.0:               return vol_label, "Above avg volume but no setup → watch closely"
        elif vol >= 1.5 and adx > 40:  return vol_label, "Strong downtrend → no entry signal"
        elif vol >= 1.5:               return vol_label, "Above avg volume but no setup → watch"
        elif vol >= 1.0:               return vol_label, "Normal activity → no setup forming yet"
        elif vol >= 0.7:               return vol_label, "Low activity → no setup forming yet"
        else:                          return vol_label, "Very low volume → no interest, wait"


# ── Rebuild ───────────────────────────────────────────────────────────────────
for idx, row in tech_df.iterrows():
    label, inference = get_volume_inference(row)
    tech_df.at[idx, 'Vol Label']     = label
    tech_df.at[idx, 'Vol Inference'] = inference

# ── Verify ────────────────────────────────────────────────────────────────────
print(f"{'Symbol':12} {'5D/15D':8} {'Label':15}  Inference")
print("─" * 90)
for _, row in tech_df.iterrows():
    print(f"  {row['Symbol']:12} {row['Vol 5D Ratio']:6.2f}x  "
          f"{row['Vol Label']:15}  {row['Vol Inference']}")

Symbol       5D/15D   Label            Inference
──────────────────────────────────────────────────────────────────────────────────────────
  TCS            0.78x  Low              Low activity → no setup forming yet
  RELIANCE       1.30x  Normal           Normal activity → no setup forming yet
  WIPRO          1.09x  Normal           Neutral volume → wait for low-volume base to form
  PERSISTENT     0.62x  Very Low         Sellers exhausted → reversal likely
  LTTS           2.35x  Very High        High volume → could be capitulation or distribution
  HAPPSTMNDS     3.69x  Extremely High   Exceptional participation → strong conviction
  CHOLAFIN       0.95x  Low              Low activity → no setup forming yet
  MUTHOOTFIN     0.87x  Low              Low activity → no setup forming yet
  MANAPPURAM     0.91x  Low              Low activity → no setup forming yet
  CREDITACC      0.76x  Low              Selling pressure easing → watch for turn
  GARFIBRES      1.51x  High             E

In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPLETE WEEKLY REPORT — FINAL WITH 5D VOLUME
# ══════════════════════════════════════════════════════════════════════════════
from datetime import datetime

today = datetime.now().strftime('%d %B %Y')
print(f"{'='*78}")
print(f"  AI STOCK SCREENER — WEEKLY REPORT")
print(f"  {today}")
print(f"{'='*78}")
print(f"\n  SecRank = rank vs same sector  |  CapRank = rank vs same sector + same cap")
print(f"  Volume  = 5-day avg vs prior 15-day avg (weekly activity measure)")

# ── TIER 1 ────────────────────────────────────────────────────────────────────
tier1 = tech_df[tech_df['Tier'].str.startswith('TIER 1')].copy()
tier1['_cap_order'] = tier1['Cap Category'].apply(lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier1 = tier1.sort_values(['_cap_order', 'Fund Score'], ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 1 — ACT NOW  ({len(tier1)} stocks)")
print(f"  Fundamentally strong + Technical setup confirmed")
print(f"{'─'*78}")

serial = 1
current_cap = None
for _, row in tier1.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML','Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_line = ''
    if row['In Consolidation'] and row['Consol Days'] > 0:
        consol_line = (f"\n    Base    : {row['Consol Days']}d "
                      f"({row['Consol Label']}) | "
                      f"{row['Pct to Breakout']:+.1f}% to breakout")

    print(f"""
  #{serial}  {row['Symbol']}  ({row['Sector']})  MCap Rs{row['Market Cap Cr']:,.0f}Cr
    Setup   : {row['Best Setup']:10}  Tech {row['Tech Score']:3.0f}/100
    Scores  : Fund {row['Fund Score']}/100  Sector Rank {row['Sector Score']}/10  Cap Rank {row['Cap Score']}/10
    Price   : Rs{row['Current Price']:.2f}  RSI {row['RSI']:.0f}  ADX {row['ADX']:.0f}  MACD {row['MACD Hist']:+.2f}{consol_line}
    Tech    : {get_reason(row)}
    Volume  : {row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) → {row['Vol Inference']}
    Sector  : {row['Sector Trend']} | {row['Sector Detail']}""")
    serial += 1

# ── TIER 2A ───────────────────────────────────────────────────────────────────
tier2a = tech_df[tech_df['Tier'] == 'TIER 2 — WATCHLIST'].copy()
tier2a = tier2a.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2A — WATCHLIST  ({len(tier2a)} stocks)")
print(f"  Setup forming — wait for confirmation before entering")
print(f"{'─'*78}")

for _, row in tier2a.iterrows():
    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
    print(f"      Technical : Setup:{row['Best Setup']:10}  Tech:{row['Tech Score']:3.0f}  "
          f"RSI:{row['RSI']:4.0f}  ADX:{row['ADX']:4.0f}  → {get_reason(row)}")
    print(f"      Volume    : {row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) → {row['Vol Inference']}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── TIER 2B ───────────────────────────────────────────────────────────────────
tier2b = tech_df[tech_df['Tier'] == 'TIER 2 — BASE BUILDING'].copy()
tier2b = tier2b.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2B — BASE BUILDING  ({len(tier2b)} stocks)")
print(f"  Long consolidation base — watch for volume breakout")
print(f"{'─'*78}")

for _, row in tier2b.iterrows():
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}  "
          f"{row['Consol Days']}d  "
          f"Range:{row['range_pct']:.1f}%  "
          f"ToBreak:{row['Pct to Breakout']:+.1f}%")
    print(f"      Technical : RSI:{row['RSI']:4.0f}  ADX:{row['ADX']:4.0f}  → {get_reason(row)}")
    print(f"      Volume    : {row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) → {row['Vol Inference']}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── TIER 3 ────────────────────────────────────────────────────────────────────
tier3 = tech_df[tech_df['Tier'] == 'TIER 3 — WAITING'].copy()
tier3['_cap_order'] = tier3['Cap Category'].apply(lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier3 = tier3.sort_values(['_cap_order', 'Fund Score'], ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 3 — WAITING  ({len(tier3)} stocks)")
print(f"  Good businesses — technical setup not ready yet")
print(f"{'─'*78}")

current_cap = None
for _, row in tier3.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML','Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}  "
          f"Vol:{row['Vol 5D Ratio']:.2f}x{consol_tag}")
    print(f"      Technical : RSI:{row['RSI']:4.0f}  ADX:{row['ADX']:4.0f}  → {get_reason(row)}")
    print(f"      Volume    : {row['Vol Label']} → {row['Vol Inference']}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── LONG BASE WATCH ───────────────────────────────────────────────────────────
consol_stocks = tech_df[tech_df['In Consolidation'] == True].copy()
consol_stocks = consol_stocks.sort_values('Consol Days', ascending=False)

print(f"\n{'─'*78}")
print(f"  LONG BASE WATCH  ({len(consol_stocks)} stocks, 300+ days)")
print(f"  Watch for breakout above range high with 2x+ volume")
print(f"{'─'*78}")
print(f"  {'Symbol':12} {'Category':14} {'Fund':5} "
      f"{'Sec':6} {'Cap':6} "
      f"{'Days':6} {'Range%':7} {'ToBreak':8} {'Vol':5}  Status")
print(f"  {'─'*85}")

for _, row in consol_stocks.iterrows():
    pct    = row['Pct to Breakout']
    status = "NEAR BREAKOUT" if pct > -5 else "Approaching" if pct > -15 else "Early stage"
    print(f"  {row['Symbol']:12} "
          f"{row['Cap Category']:14} "
          f"{row['Fund Score']:4.1f} "
          f"{row['Sector Score']:5.1f} "
          f"{row['Cap Score']:5.1f} "
          f"{row['Consol Days']:5}d "
          f"{row['range_pct']:6.1f}% "
          f"{pct:+7.1f}% "
          f"{row['Vol 5D Ratio']:.2f}x  "
          f"{status}")

print(f"\n{'─'*78}")
print(f"  L=Large Cap  ML=Mini Large Cap  M=Mid Cap  S=Small Cap")
print(f"  SecRank = rank vs same sector | CapRank = rank vs same sector + cap")
print(f"  Volume  = 5-day avg vs prior 15-day avg")
print(f"{'─'*78}")

tech_df.to_csv('../data/technical_report.csv', index=False)
print("\nSaved: data/technical_report.csv ✅")

  AI STOCK SCREENER — WEEKLY REPORT
  21 March 2026

  SecRank = rank vs same sector  |  CapRank = rank vs same sector + same cap
  Volume  = 5-day avg vs prior 15-day avg (weekly activity measure)


NameError: name 'CAP_ORDER' is not defined

In [14]:
# ── Improved sector trend classification ──────────────────────────────────────
def get_sector_trend(sector):
    if sector not in sector_price:
        return "No Data", "N/A"

    df     = calculate_sector_indicators(sector_price[sector])
    latest = df.iloc[-1]

    close    = latest['Close']
    rsi      = round(latest['RSI'],      1)
    adx      = round(latest['ADX'],      1)
    ema200   = latest['EMA200']
    ema50    = latest['EMA50']
    di_plus  = latest['DI_Plus']
    di_minus = latest['DI_Minus']

    above_ema200 = close > ema200
    above_ema50  = close > ema50
    uptrend      = di_plus > di_minus

    # ── Classify into 6 clear categories ──────────────────────────────────────
    if above_ema200 and above_ema50 and adx > 30 and uptrend and rsi > 55:
        label = "Strong Uptrend ↑↑"
    elif above_ema200 and uptrend and adx > 20 and rsi > 50:
        label = "Uptrend ↑"
    elif above_ema200 and adx <= 20:
        label = "Weak Uptrend / Sideways →↑"
    elif not above_ema200 and adx <= 20 and rsi >= 45:
        label = "Sideways / Weak →"
    elif not above_ema200 and adx > 20 and not uptrend and rsi < 45:
        label = "Downtrend ↓"
    elif not above_ema200 and adx > 30 and not uptrend and rsi < 40:
        label = "Strong Downtrend ↓↓"
    else:
        label = "Weak Downtrend →↓"

    detail = f"RSI {rsi} | ADX {adx} | {'Above' if above_ema200 else 'Below'} EMA200"
    return label, detail


# ── Rebuild sector trend for all stocks ───────────────────────────────────────
for idx, row in tech_df.iterrows():
    label, detail = get_sector_trend(row['Sector'])
    tech_df.at[idx, 'Sector Trend']  = label
    tech_df.at[idx, 'Sector Detail'] = detail

# ── Verify ────────────────────────────────────────────────────────────────────
print(f"{'Sector':40} {'Trend':28} Detail")
print("─" * 90)
for sector in sorted(tech_df['Sector'].unique()):
    label, detail = get_sector_trend(sector)
    print(f"  {sector:40} {label:28} {detail}")

Sector                                   Trend                        Detail
──────────────────────────────────────────────────────────────────────────────────────────
  Automobile and Auto Components           Downtrend ↓                  RSI 37.2 | ADX 40.3 | Below EMA200
  Capital Goods                            Downtrend ↓                  RSI 36.0 | ADX 40.7 | Below EMA200
  Chemicals                                Downtrend ↓                  RSI 34.4 | ADX 42.3 | Below EMA200
  Consumer Durables                        Downtrend ↓                  RSI 36.4 | ADX 46.3 | Below EMA200
  Consumer Services                        Downtrend ↓                  RSI 34.4 | ADX 42.3 | Below EMA200
  Financial Services                       Downtrend ↓                  RSI 32.5 | ADX 28.2 | Below EMA200
  Healthcare                               Weak Downtrend →↓            RSI 45.0 | ADX 27.7 | Above EMA200
  Information Technology                   Downtrend ↓                  RSI 40.4 | 

In [15]:
def get_sector_trend(sector):
    if sector not in sector_price:
        return "No Data", "N/A"

    df     = calculate_sector_indicators(sector_price[sector])
    latest = df.iloc[-1]

    close    = latest['Close']
    rsi      = round(latest['RSI'],      1)
    adx      = round(latest['ADX'],      1)
    ema20    = latest['EMA20']
    ema50    = latest['EMA50']
    ema200   = latest['EMA200']
    di_plus  = latest['DI_Plus']
    di_minus = latest['DI_Minus']
    macd     = latest['MACD']
    signal   = latest['Signal']
    macd_h   = latest['MACD_Hist']

    above_ema200 = close > ema200
    above_ema50  = close > ema50
    above_ema20  = close > ema20
    ema_aligned  = ema20 > ema50 > ema200   # full bull alignment
    ema_bearish  = ema20 < ema50 < ema200   # full bear alignment
    uptrend      = di_plus > di_minus
    macd_bull    = macd > signal and macd_h > 0
    macd_bear    = macd < signal and macd_h < 0
    macd_turning = macd_h > 0 and macd < signal  # histogram turning up

    # ── Score bullish signals (0-6) ───────────────────────────────────────────
    bull_score = sum([
        above_ema200,        # price above long term MA
        above_ema50,         # price above mid term MA
        ema_aligned,         # EMAs in full bull order
        uptrend,             # DI+ > DI-
        macd_bull,           # MACD above signal, hist positive
        rsi > 55,            # RSI in bullish zone
    ])

    # ── Score bearish signals (0-6) ───────────────────────────────────────────
    bear_score = sum([
        not above_ema200,    # price below long term MA
        not above_ema50,     # price below mid term MA
        ema_bearish,         # EMAs in full bear order
        not uptrend,         # DI- > DI+
        macd_bear,           # MACD below signal, hist negative
        rsi < 45,            # RSI in bearish zone
    ])

    # ── Classify using scores + ADX (trend strength) ──────────────────────────
    if bull_score >= 5 and adx > 25:
        label = "Strong Uptrend ↑↑"
    elif bull_score >= 4 and adx > 20:
        label = "Uptrend ↑"
    elif bull_score >= 3 and adx <= 20:
        label = "Weak Uptrend →↑"
    elif bull_score == bear_score or adx < 15:
        label = "Sideways →"
    elif bear_score >= 5 and adx > 25:
        label = "Strong Downtrend ↓↓"
    elif bear_score >= 4 and adx > 20:
        label = "Downtrend ↓"
    elif bear_score >= 3 and adx <= 20:
        label = "Weak Downtrend →↓"
    else:
        label = "Sideways →"

    # ── Add MACD turning signal ────────────────────────────────────────────────
    if macd_turning and bear_score >= 3:
        label += " (MACD turning)"

    detail = (f"RSI {rsi} | ADX {adx} | MACD {'▲' if macd_bull else '▼' if macd_bear else '~'} | "
              f"EMA {'↑↑↑' if ema_aligned else '↓↓↓' if ema_bearish else 'mixed'} | "
              f"{'Above' if above_ema200 else 'Below'} EMA200")

    return label, detail


# ── Rebuild sector trend for all stocks ───────────────────────────────────────
for idx, row in tech_df.iterrows():
    label, detail = get_sector_trend(row['Sector'])
    tech_df.at[idx, 'Sector Trend']  = label
    tech_df.at[idx, 'Sector Detail'] = detail

# ── Verify ────────────────────────────────────────────────────────────────────
print(f"{'Sector':40} {'Trend':35} Detail")
print("─" * 105)
for sector in sorted(tech_df['Sector'].unique()):
    label, detail = get_sector_trend(sector)
    print(f"  {sector:40} {label:35} {detail}")

Sector                                   Trend                               Detail
─────────────────────────────────────────────────────────────────────────────────────────────────────────
  Automobile and Auto Components           Strong Downtrend ↓↓                 RSI 37.2 | ADX 40.3 | MACD ▼ | EMA mixed | Below EMA200
  Capital Goods                            Strong Downtrend ↓↓                 RSI 36.0 | ADX 40.7 | MACD ▼ | EMA mixed | Below EMA200
  Chemicals                                Strong Downtrend ↓↓                 RSI 34.4 | ADX 42.3 | MACD ▼ | EMA mixed | Below EMA200
  Consumer Durables                        Strong Downtrend ↓↓                 RSI 36.4 | ADX 46.3 | MACD ▼ | EMA ↓↓↓ | Below EMA200
  Consumer Services                        Strong Downtrend ↓↓                 RSI 34.4 | ADX 42.3 | MACD ▼ | EMA mixed | Below EMA200
  Financial Services                       Strong Downtrend ↓↓                 RSI 32.5 | ADX 28.2 | MACD ▼ | EMA mixed | Below EMA200
  

In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPLETE WEEKLY REPORT — ALL FIXES APPLIED
# ══════════════════════════════════════════════════════════════════════════════
from datetime import datetime

today = datetime.now().strftime('%d %B %Y')
print(f"{'='*78}")
print(f"  AI STOCK SCREENER — WEEKLY REPORT")
print(f"  {today}")
print(f"{'='*78}")
print(f"\n  SecRank = rank vs same sector  |  CapRank = rank vs same sector + same cap")
print(f"  Volume  = 5-day avg vs prior 15-day avg")

# ── TIER 1 ────────────────────────────────────────────────────────────────────
tier1 = tech_df[tech_df['Tier'].str.startswith('TIER 1')].copy()
tier1['_cap_order'] = tier1['Cap Category'].apply(
    lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier1 = tier1.sort_values(['_cap_order', 'Fund Score'],
                          ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 1 — ACT NOW  ({len(tier1)} stocks)")
print(f"  Fundamentally strong + Technical setup confirmed")
print(f"{'─'*78}")

serial = 1
current_cap = None
for _, row in tier1.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_line = ''
    if row['In Consolidation'] and row['Consol Days'] > 0:
        consol_line = (f"\n    Base    : {row['Consol Days']}d "
                      f"({row['Consol Label']}) | "
                      f"{row['Pct to Breakout']:+.1f}% to breakout")

    # Price moved to name line, Price row removed
    print(f"""
  #{serial}  {row['Symbol']}  ({row['Sector']})  MCap Rs{row['Market Cap Cr']:,.0f}Cr  Rs{row['Current Price']:.2f}
    Setup   : {row['Best Setup']:10}  Tech {row['Tech Score']:3.0f}/100
    Scores  : Fund {row['Fund Score']}/100  Sector Rank {row['Sector Score']}/10  Cap Rank {row['Cap Score']}/10{consol_line}
    Tech    : RSI {row['RSI']:.0f}  ADX {row['ADX']:.0f}  MACD {row['MACD Hist']:+.2f}  → {get_reason(row)}
    Volume  : {row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) → {row['Vol Inference']}
    Sector  : {row['Sector Trend']} | {row['Sector Detail']}""")
    serial += 1

# ── TIER 2A ───────────────────────────────────────────────────────────────────
tier2a = tech_df[tech_df['Tier'] == 'TIER 2 — WATCHLIST'].copy()
tier2a = tier2a.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2A — WATCHLIST  ({len(tier2a)} stocks)")
print(f"  Setup forming — wait for confirmation before entering")
print(f"{'─'*78}")

for _, row in tier2a.iterrows():
    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''
    # Price added to first line
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
    print(f"      Technical : Setup:{row['Best Setup']:10}  Tech:{row['Tech Score']:3.0f}  "
          f"RSI:{row['RSI']:4.0f}  ADX:{row['ADX']:4.0f}  "
          f"MACD:{row['MACD Hist']:+.2f}  → {get_reason(row)}")
    print(f"      Volume    : {row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) "
          f"→ {row['Vol Inference']}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── TIER 2B ───────────────────────────────────────────────────────────────────
tier2b = tech_df[tech_df['Tier'] == 'TIER 2 — BASE BUILDING'].copy()
tier2b = tier2b.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2B — BASE BUILDING  ({len(tier2b)} stocks)")
print(f"  Long consolidation base — watch for volume breakout")
print(f"{'─'*78}")

for _, row in tier2b.iterrows():
    # Price added to first line
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}  "
          f"{row['Consol Days']}d  "
          f"Range:{row['range_pct']:.1f}%  "
          f"ToBreak:{row['Pct to Breakout']:+.1f}%")
    print(f"      Technical : RSI:{row['RSI']:4.0f}  ADX:{row['ADX']:4.0f}  "
          f"MACD:{row['MACD Hist']:+.2f}  → {get_reason(row)}")
    print(f"      Volume    : {row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) "
          f"→ {row['Vol Inference']}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── TIER 3 ────────────────────────────────────────────────────────────────────
tier3 = tech_df[tech_df['Tier'] == 'TIER 3 — WAITING'].copy()
tier3['_cap_order'] = tier3['Cap Category'].apply(
    lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier3 = tier3.sort_values(['_cap_order', 'Fund Score'],
                          ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 3 — WAITING  ({len(tier3)} stocks)")
print(f"  Good businesses — technical setup not ready yet")
print(f"{'─'*78}")

current_cap = None
for _, row in tier3.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''
    # Price in first line, Vol moved to Volume row
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
    print(f"      Technical : RSI:{row['RSI']:4.0f}  ADX:{row['ADX']:4.0f}  "
          f"MACD:{row['MACD Hist']:+.2f}  → {get_reason(row)}")
    print(f"      Volume    : {row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) "
          f"→ {row['Vol Inference']}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── LONG BASE WATCH ───────────────────────────────────────────────────────────
consol_stocks = tech_df[tech_df['In Consolidation'] == True].copy()
consol_stocks = consol_stocks.sort_values('Consol Days', ascending=False)

print(f"\n{'─'*78}")
print(f"  LONG BASE WATCH  ({len(consol_stocks)} stocks, 300+ days)")
print(f"  Watch for breakout above range high with 2x+ volume")
print(f"{'─'*78}")
print(f"  {'Symbol':12} {'Category':14} {'Fund':5} "
      f"{'Sec':6} {'Cap':6} "
      f"{'Days':6} {'Range%':7} {'ToBreak':8} {'Vol':6}  Status")
print(f"  {'─'*85}")

for _, row in consol_stocks.iterrows():
    pct    = row['Pct to Breakout']
    status = "NEAR BREAKOUT" if pct > -5 else \
             "Approaching"   if pct > -15 else "Early stage"
    print(f"  {row['Symbol']:12} "
          f"{row['Cap Category']:14} "
          f"{row['Fund Score']:4.1f} "
          f"{row['Sector Score']:5.1f} "
          f"{row['Cap Score']:5.1f} "
          f"{row['Consol Days']:5}d "
          f"{row['range_pct']:6.1f}% "
          f"{pct:+7.1f}% "
          f"{row['Vol 5D Ratio']:.2f}x  "
          f"{status}")

print(f"\n{'─'*78}")
print(f"  L=Large Cap  ML=Mini Large Cap  M=Mid Cap  S=Small Cap")
print(f"  SecRank = rank vs same sector | CapRank = rank vs same sector + cap")
print(f"  Volume  = 5-day avg vs prior 15-day avg")
print(f"{'─'*78}")

tech_df.to_csv('../data/technical_report.csv', index=False)
print("\nSaved: data/technical_report.csv ✅")

  AI STOCK SCREENER — WEEKLY REPORT
  21 March 2026

  SecRank = rank vs same sector  |  CapRank = rank vs same sector + same cap
  Volume  = 5-day avg vs prior 15-day avg


NameError: name 'CAP_ORDER' is not defined

In [17]:
# ── Calculate Breakout Vol = Week vol vs Consolidation avg vol ────────────────
print("Calculating breakout volume signal...")

breakout_vol_data = {}

for _, crow in consol_info_df.iterrows():
    symbol      = crow['Symbol']
    consol_days = int(crow['consol_days'])

    if symbol not in price_data:
        continue

    df = price_data[symbol].copy()

    if len(df) < consol_days + 5:
        breakout_vol_data[symbol] = 1.0
        continue

    # Week vol = last 5 days average
    week_vol   = df['Volume'].iloc[-5:].mean()

    # Consolidation avg = average over entire consolidation period
    # excluding last 5 days to avoid overlap
    consol_avg = df['Volume'].iloc[-(consol_days):-5].mean()

    ratio = week_vol / consol_avg if consol_avg > 0 else 1.0
    breakout_vol_data[symbol] = round(ratio, 2)

    print(f"  {symbol:12}  "
          f"Week vol:{week_vol:>12,.0f}  "
          f"Consol avg:{consol_avg:>12,.0f}  "
          f"Breakout Vol:{ratio:.2f}x  "
          f"{'⚡ BUILDING' if ratio >= 1.5 else '→ quiet' if ratio < 0.8 else '~ normal'}")

print(f"\nDone: {len(breakout_vol_data)} consolidating stocks")

Calculating breakout volume signal...
  FINEORG       Week vol:      13,475  Consol avg:      30,838  Breakout Vol:0.44x  → quiet
  RELIANCE      Week vol:  20,598,694  Consol avg:  12,182,786  Breakout Vol:1.69x  ⚡ BUILDING
  SUNPHARMA     Week vol:   3,016,522  Consol avg:   2,330,102  Breakout Vol:1.29x  ~ normal
  SUPRAJIT      Week vol:      62,251  Consol avg:     200,907  Breakout Vol:0.31x  → quiet
  HAL           Week vol:   1,578,525  Consol avg:   1,968,219  Breakout Vol:0.80x  ~ normal
  IPCALAB       Week vol:     241,624  Consol avg:     371,792  Breakout Vol:0.65x  → quiet
  GRANULES      Week vol:     512,299  Consol avg:   1,869,613  Breakout Vol:0.27x  → quiet
  PERSISTENT    Week vol:     621,244  Consol avg:     545,666  Breakout Vol:1.14x  ~ normal

Done: 8 consolidating stocks


In [18]:
# ── Add Breakout Vol to tech_df and verify ────────────────────────────────────
tech_df['Breakout Vol'] = tech_df['Symbol'].map(breakout_vol_data).fillna(0.0)

print("Breakout Vol added:")
print(tech_df[['Symbol', 'In Consolidation', 'Breakout Vol']].to_string(index=False))

Breakout Vol added:
    Symbol  In Consolidation  Breakout Vol
       TCS             False          0.00
  RELIANCE              True          1.69
     WIPRO             False          0.00
PERSISTENT              True          1.14
      LTTS             False          0.00
HAPPSTMNDS             False          0.00
  CHOLAFIN             False          0.00
MUTHOOTFIN             False          0.00
MANAPPURAM             False          0.00
 CREDITACC             False          0.00
 GARFIBRES             False          0.00
  SUPRAJIT              True          0.31
HIMATSEIDE             False          0.00
       HAL              True          0.80
       BEL             False          0.00
   MIDHANI             False          0.00
     PARAS             False          0.00
VINATIORGA             False          0.00
   FINEORG              True          0.44
GALAXYSURF             False          0.00
     AAVAS             False          0.00
 SUNPHARMA              True      

In [19]:
# ── Breakout vol inference ─────────────────────────────────────────────────────
def get_breakout_vol_inference(week_vol, break_vol):
    if break_vol >= 2.0:
        return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ⚡⚡ → Strong breakout volume building"
    elif break_vol >= 1.5:
        return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ⚡ → Volume building toward breakout"
    elif break_vol >= 1.0:
        return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ~ → Normal volume, no breakout pressure"
    elif break_vol >= 0.7:
        return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x → Quiet, below consolidation avg"
    else:
        return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x → Very quiet, no breakout interest"


# ══════════════════════════════════════════════════════════════════════════════
# COMPLETE WEEKLY REPORT — FINAL VERSION
# ══════════════════════════════════════════════════════════════════════════════
from datetime import datetime

today = datetime.now().strftime('%d %B %Y')
print(f"{'='*78}")
print(f"  AI STOCK SCREENER — WEEKLY REPORT")
print(f"  {today}")
print(f"{'='*78}")
print(f"\n  SecRank  = rank vs same sector  |  CapRank = rank vs same sector + same cap")
print(f"  Week Vol = this week avg vs prior 15-day avg")
print(f"  Break Vol= this week avg vs consolidation period avg (breakout signal)")

# ── TIER 1 ────────────────────────────────────────────────────────────────────
tier1 = tech_df[tech_df['Tier'].str.startswith('TIER 1')].copy()
tier1['_cap_order'] = tier1['Cap Category'].apply(
    lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier1 = tier1.sort_values(['_cap_order', 'Fund Score'],
                          ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 1 — ACT NOW  ({len(tier1)} stocks)")
print(f"  Fundamentally strong + Technical setup confirmed")
print(f"{'─'*78}")

serial = 1
current_cap = None
for _, row in tier1.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_line = ''
    if row['In Consolidation'] and row['Consol Days'] > 0:
        bvol_inf = get_breakout_vol_inference(
            row['Vol 5D Ratio'], row['Breakout Vol'])
        consol_line = (f"\n    Base    : {row['Consol Days']}d "
                      f"({row['Consol Label']}) | "
                      f"{row['Pct to Breakout']:+.1f}% to breakout"
                      f"\n    Break   : {bvol_inf}")

    print(f"""
  #{serial}  {row['Symbol']}  ({row['Sector']})  MCap Rs{row['Market Cap Cr']:,.0f}Cr  Rs{row['Current Price']:.2f}
    Setup   : {row['Best Setup']:10}  Tech {row['Tech Score']:3.0f}/100
    Scores  : Fund {row['Fund Score']}/100  Sector Rank {row['Sector Score']}/10  Cap Rank {row['Cap Score']}/10{consol_line}
    Tech    : RSI {row['RSI']:.0f}  ADX {row['ADX']:.0f}  MACD {row['MACD Hist']:+.2f}  → {get_reason(row)}
    Volume  : Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) → {row['Vol Inference']}
    Sector  : {row['Sector Trend']} | {row['Sector Detail']}""")
    serial += 1

# ── TIER 2A ───────────────────────────────────────────────────────────────────
tier2a = tech_df[tech_df['Tier'] == 'TIER 2 — WATCHLIST'].copy()
tier2a = tier2a.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2A — WATCHLIST  ({len(tier2a)} stocks)")
print(f"  Setup forming — wait for confirmation before entering")
print(f"{'─'*78}")

for _, row in tier2a.iterrows():
    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
    print(f"      Technical : [{row['Best Setup']}|{row['Tech Score']:.0f}]  "
          f"RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
    print(f"      Volume    : Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) "
          f"→ {row['Vol Inference']}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── TIER 2B ───────────────────────────────────────────────────────────────────
tier2b = tech_df[tech_df['Tier'] == 'TIER 2 — BASE BUILDING'].copy()
tier2b = tier2b.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2B — BASE BUILDING  ({len(tier2b)} stocks)")
print(f"  Long consolidation base — watch for volume breakout")
print(f"{'─'*78}")

for _, row in tier2b.iterrows():
    bvol_inf = get_breakout_vol_inference(
        row['Vol 5D Ratio'], row['Breakout Vol'])
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}  "
          f"{row['Consol Days']}d  "
          f"Range:{row['range_pct']:.1f}%  "
          f"ToBreak:{row['Pct to Breakout']:+.1f}%")
    print(f"      Technical : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
    print(f"      Volume    : {bvol_inf}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── TIER 3 ────────────────────────────────────────────────────────────────────
tier3 = tech_df[tech_df['Tier'] == 'TIER 3 — WAITING'].copy()
tier3['_cap_order'] = tier3['Cap Category'].apply(
    lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier3 = tier3.sort_values(['_cap_order', 'Fund Score'],
                          ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 3 — WAITING  ({len(tier3)} stocks)")
print(f"  Good businesses — technical setup not ready yet")
print(f"{'─'*78}")

current_cap = None
for _, row in tier3.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''

    # For consolidating stocks in Tier 3 — show both volumes
    if row['In Consolidation'] and row['Breakout Vol'] > 0:
        vol_line = get_breakout_vol_inference(
            row['Vol 5D Ratio'], row['Breakout Vol'])
    else:
        vol_line = (f"Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) "
                   f"→ {row['Vol Inference']}")

    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
    print(f"      Technical : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
    print(f"      Volume    : {vol_line}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── LONG BASE WATCH ───────────────────────────────────────────────────────────
consol_stocks = tech_df[tech_df['In Consolidation'] == True].copy()
consol_stocks = consol_stocks.sort_values('Consol Days', ascending=False)

print(f"\n{'─'*78}")
print(f"  LONG BASE WATCH  ({len(consol_stocks)} stocks, 300+ days)")
print(f"  Watch for breakout above range high with 2x+ volume")
print(f"{'─'*78}")
print(f"  {'Symbol':12} {'Category':14} {'Fund':5} "
      f"{'Sec':6} {'Cap':6} "
      f"{'Days':6} {'Range%':7} {'ToBreak':8}  Volume Signal")
print(f"  {'─'*95}")

for _, row in consol_stocks.iterrows():
    pct      = row['Pct to Breakout']
    status   = "NEAR BREAKOUT" if pct > -5 else \
               "Approaching"   if pct > -15 else "Early stage"
    bvol_inf = get_breakout_vol_inference(
        row['Vol 5D Ratio'], row['Breakout Vol'])
    print(f"  {row['Symbol']:12} "
          f"{row['Cap Category']:14} "
          f"{row['Fund Score']:4.1f} "
          f"{row['Sector Score']:5.1f} "
          f"{row['Cap Score']:5.1f} "
          f"{row['Consol Days']:5}d "
          f"{row['range_pct']:6.1f}% "
          f"{pct:+7.1f}%  "
          f"{bvol_inf}  [{status}]")

print(f"\n{'─'*78}")
print(f"  L=Large Cap  ML=Mini Large Cap  M=Mid Cap  S=Small Cap")
print(f"  SecRank  = rank vs same sector | CapRank = rank vs same sector + cap")
print(f"  Week Vol = this week avg vs prior 15-day avg")
print(f"  Break Vol= this week avg vs consolidation period avg")
print(f"{'─'*78}")

tech_df.to_csv('../data/technical_report.csv', index=False)
print("\nSaved: data/technical_report.csv ✅")

  AI STOCK SCREENER — WEEKLY REPORT
  21 March 2026

  SecRank  = rank vs same sector  |  CapRank = rank vs same sector + same cap
  Week Vol = this week avg vs prior 15-day avg
  Break Vol= this week avg vs consolidation period avg (breakout signal)


NameError: name 'CAP_ORDER' is not defined

In [20]:
# ── LONG BASE WATCH — compact single line ─────────────────────────────────────
consol_stocks = tech_df[tech_df['In Consolidation'] == True].copy()
consol_stocks = consol_stocks.sort_values('Consol Days', ascending=False)

print(f"\n{'─'*78}")
print(f"  LONG BASE WATCH  ({len(consol_stocks)} stocks, 300+ days)")
print(f"  Watch for breakout above range high — Break Vol > 1.5x = signal building")
print(f"{'─'*78}")
print(f"  {'Symbol':12} {'Category':14} {'Days':6} {'Range%':7} {'ToBreak':8} "
      f"{'WkVol':7} {'BrkVol':7}  Signal & Status")
print(f"  {'─'*90}")

for _, row in consol_stocks.iterrows():
    pct      = row['Pct to Breakout']
    status   = "NEAR BREAKOUT ⚡" if pct > -5 else \
               "Approaching"      if pct > -15 else "Early stage"
    bvol     = row['Breakout Vol']
    wvol     = row['Vol 5D Ratio']
    bvol_tag = "⚡ BUILDING" if bvol >= 1.5 else "~ normal" if bvol >= 0.8 else "→ quiet"

    print(f"  {row['Symbol']:12} "
          f"{row['Cap Category']:14} "
          f"{row['Consol Days']:5}d "
          f"{row['range_pct']:6.1f}% "
          f"{pct:+7.1f}%  "
          f"{wvol:.2f}x  "
          f"{bvol:.2f}x  "
          f"{bvol_tag:12}  [{status}]")

print(f"\n{'─'*78}")
print(f"  L=Large Cap  ML=Mini Large Cap  M=Mid Cap  S=Small Cap")
print(f"  SecRank  = rank vs same sector | CapRank = rank vs same sector + cap")
print(f"  Week Vol = this week avg vs prior 15-day avg")
print(f"  Break Vol= this week avg vs consolidation period avg")
print(f"{'─'*78}")

tech_df.to_csv('../data/technical_report.csv', index=False)
print("\nSaved: data/technical_report.csv ✅")


──────────────────────────────────────────────────────────────────────────────
  LONG BASE WATCH  (8 stocks, 300+ days)
  Watch for breakout above range high — Break Vol > 1.5x = signal building
──────────────────────────────────────────────────────────────────────────────
  Symbol       Category       Days   Range%  ToBreak  WkVol   BrkVol   Signal & Status
  ──────────────────────────────────────────────────────────────────────────────────────────
  FINEORG      Mid Cap          798d   37.4%   -21.6%  0.50x  0.44x  → quiet       [Early stage]
  RELIANCE     Large Cap        606d   34.9%   -11.5%  1.30x  1.69x  ⚡ BUILDING    [Approaching]
  SUNPHARMA    Large Cap        520d   26.1%    -2.7%  1.23x  1.29x  ~ normal      [NEAR BREAKOUT ⚡]
  SUPRAJIT     Mid Cap          491d   39.0%   -24.2%  0.73x  0.31x  → quiet       [Early stage]
  HAL          Large Cap        484d   49.8%   -23.9%  0.83x  0.80x  ~ normal      [Early stage]
  IPCALAB      Mini Large Cap   481d   37.8%    -3.2%  1

In [21]:
# ── Save final tech_df with all new columns ───────────────────────────────────
tech_df.to_csv('../data/technical_report.csv', index=False)

# ── Save sector data for reuse ────────────────────────────────────────────────
import pickle
with open('../data/sector_index_data.pkl', 'wb') as f:
    pickle.dump(sector_price, f)

# ── Confirm all columns saved ─────────────────────────────────────────────────
print("Saved files:")
print("  data/technical_report.csv")
print("  data/sector_index_data.pkl")
print(f"\nColumns in technical_report.csv ({len(tech_df.columns)}):")
print(tech_df.columns.tolist())

Saved files:
  data/technical_report.csv
  data/sector_index_data.pkl

Columns in technical_report.csv (34):
['Symbol', 'Sector', 'Fund Score', 'Current Price', 'RSI', 'ADX', 'MACD Hist', 'Vol Ratio', 'EMA20', 'EMA50', 'EMA200', 'DI+', 'DI-', 'Momentum Score', 'Reversal Score', 'Best Setup', 'Tech Score', 'Tier', 'In Consolidation', 'Consol Days', 'Consol Label', 'Pct to Breakout', 'Consol Volume', 'Sector Score', 'Cap Score', 'Market Cap Cr', 'Cap Category', 'Vol Label', 'Vol Inference', 'Sector Trend', 'Sector Detail', 'range_pct', 'Vol 5D Ratio', 'Breakout Vol']


In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPLETE WEEKLY REPORT — SELF CONTAINED
# ══════════════════════════════════════════════════════════════════════════════
from datetime import datetime

# ── Helper functions ──────────────────────────────────────────────────────────
CAP_ORDER = ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']

def get_reason(row):
    rsi    = float(row['RSI'])
    adx    = float(row['ADX'])
    setup  = str(row['Best Setup'])
    hist   = float(row['MACD Hist'])
    di_p   = float(row.get('DI+', 0))
    di_m   = float(row.get('DI-', 0))
    ema50  = float(row.get('EMA50',  0))
    ema200 = float(row.get('EMA200', 0))
    price  = float(row.get('Current Price', 0))

    uptrend   = (adx > 25 and di_p > di_m and price > ema50 and price > ema200)
    bounce    = (price > ema50 and price < ema200)
    downtrend = (price < ema200) or (adx > 25 and di_m > di_p)

    if setup == 'Momentum':
        if uptrend:                        return f"Uptrend confirmed, RSI {rsi:.0f}, ADX {adx:.0f}"
        elif bounce:                       return f"Short bounce, below EMA200"
        elif price < ema200 and rsi > 50:  return f"RSI {rsi:.0f} bounce in downtrend"
        elif rsi >= 50:                    return f"Momentum building, RSI {rsi:.0f}"
        return f"Momentum setup, RSI {rsi:.0f}"

    elif setup == 'Reversal':
        if rsi < 35 and hist > 0:          return f"Oversold RSI {rsi:.0f}, MACD turning up"
        elif rsi < 35:                     return f"Oversold RSI {rsi:.0f}, watch for turn"
        elif hist > 0 and downtrend:       return f"Downtrend weakening, MACD improving"
        return f"Divergence forming, RSI {rsi:.0f}"

    else:
        if downtrend and adx > 40:         return f"Strong downtrend ADX {adx:.0f}, wait"
        elif downtrend and adx > 25:       return f"Downtrend ADX {adx:.0f}, no setup yet"
        elif bounce:                       return f"Bounce in downtrend, below EMA200"
        elif rsi < 25:                     return f"Deeply oversold RSI {rsi:.0f}, too early"
        elif rsi < 40:                     return f"Oversold RSI {rsi:.0f}, watch for turn"
        elif rsi > 60 and not uptrend:     return f"RSI {rsi:.0f} bounce, below EMA200"
        else:                              return f"RSI {rsi:.0f}, setup not confirmed"


def get_volume_inference(row):
    vol   = float(row['Vol 5D Ratio'])
    setup = str(row['Best Setup'])
    rsi   = float(row['RSI'])
    adx   = float(row['ADX'])

    if   vol >= 3.0:  vol_label = "Extremely High"
    elif vol >= 2.0:  vol_label = "Very High"
    elif vol >= 1.5:  vol_label = "High"
    elif vol >= 1.0:  vol_label = "Normal"
    elif vol >= 0.7:  vol_label = "Low"
    else:             vol_label = "Very Low"

    if setup == 'Momentum':
        if   vol >= 3.0: return vol_label, "Exceptional participation → strong conviction"
        elif vol >= 2.0: return vol_label, "Strong participation → trend likely to continue"
        elif vol >= 1.5: return vol_label, "Good volume support → momentum confirmed"
        elif vol >= 1.0: return vol_label, "Average volume → monitor for increase"
        elif vol >= 0.7: return vol_label, "Weak volume → momentum may fade, wait"
        else:            return vol_label, "Very low volume → no conviction, caution"

    elif setup == 'Reversal':
        if   vol >= 3.0: return vol_label, "Very high volume on reversal → strong capitulation signal"
        elif vol >= 2.0: return vol_label, "High volume → could be capitulation or distribution"
        elif vol >= 1.5: return vol_label, "Elevated volume on reversal → watch direction carefully"
        elif vol >= 1.0: return vol_label, "Neutral volume → wait for low-volume base to form"
        elif vol >= 0.7: return vol_label, "Selling pressure easing → watch for turn"
        else:            return vol_label, "Sellers exhausted → reversal likely"

    else:
        if   vol >= 3.0 and adx > 40:  return vol_label, "Very high volume downtrend → capitulation possible"
        elif vol >= 3.0:               return vol_label, "Exceptional volume → major event this week, watch"
        elif vol >= 2.0 and adx > 40:  return vol_label, "High volume downtrend → stay away"
        elif vol >= 2.0 and rsi < 35:  return vol_label, "High volume selloff → possible capitulation"
        elif vol >= 2.0:               return vol_label, "Above avg volume but no setup → watch closely"
        elif vol >= 1.5 and adx > 40:  return vol_label, "Strong downtrend → no entry signal"
        elif vol >= 1.5:               return vol_label, "Above avg volume but no setup → watch"
        elif vol >= 1.0:               return vol_label, "Normal activity → no setup forming yet"
        elif vol >= 0.7:               return vol_label, "Low activity → no setup forming yet"
        else:                          return vol_label, "Very low volume → no interest, wait"


def get_breakout_vol_inference(week_vol, break_vol):
    if   break_vol >= 2.0: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ⚡⚡ → Strong breakout volume building"
    elif break_vol >= 1.5: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ⚡  → Volume building toward breakout"
    elif break_vol >= 1.0: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ~  → Normal volume, no breakout pressure"
    elif break_vol >= 0.7: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x    → Quiet, below consolidation avg"
    else:                  return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x    → Very quiet, no breakout interest"


def get_sector_trend(sector):
    if sector not in sector_price:
        return "No Data", "N/A"

    df     = calculate_sector_indicators(sector_price[sector])
    latest = df.iloc[-1]

    close    = latest['Close']
    rsi      = round(latest['RSI'],   1)
    adx      = round(latest['ADX'],   1)
    ema20    = latest['EMA20']
    ema50    = latest['EMA50']
    ema200   = latest['EMA200']
    di_plus  = latest['DI_Plus']
    di_minus = latest['DI_Minus']
    macd     = latest['MACD']
    signal   = latest['Signal']
    macd_h   = latest['MACD_Hist']

    above_ema200 = close > ema200
    above_ema50  = close > ema50
    ema_aligned  = ema20 > ema50 > ema200
    ema_bearish  = ema20 < ema50 < ema200
    uptrend      = di_plus > di_minus
    macd_bull    = macd > signal and macd_h > 0
    macd_bear    = macd < signal and macd_h < 0
    macd_turning = macd_h > 0 and macd < signal

    bull_score = sum([above_ema200, above_ema50, ema_aligned,
                      uptrend, macd_bull, rsi > 55])
    bear_score = sum([not above_ema200, not above_ema50, ema_bearish,
                      not uptrend, macd_bear, rsi < 45])

    if   bull_score >= 5 and adx > 25: label = "Strong Uptrend ↑↑"
    elif bull_score >= 4 and adx > 20: label = "Uptrend ↑"
    elif bull_score >= 3 and adx <= 20: label = "Weak Uptrend →↑"
    elif bull_score == bear_score or adx < 15: label = "Sideways →"
    elif bear_score >= 5 and adx > 25: label = "Strong Downtrend ↓↓"
    elif bear_score >= 4 and adx > 20: label = "Downtrend ↓"
    elif bear_score >= 3 and adx <= 20: label = "Weak Downtrend →↓"
    else: label = "Sideways →"

    if macd_turning and bear_score >= 3:
        label += " (MACD turning)"

    ema_str  = '↑↑↑' if ema_aligned else '↓↓↓' if ema_bearish else 'mixed'
    macd_str = '▲' if macd_bull else '▼' if macd_bear else '~'
    detail   = (f"RSI {rsi} | ADX {adx} | MACD {macd_str} | "
                f"EMA {ema_str} | {'Above' if above_ema200 else 'Below'} EMA200")

    return label, detail


# ── Rebuild all inference columns fresh ───────────────────────────────────────
for idx, row in tech_df.iterrows():
    label, inference = get_volume_inference(row)
    tech_df.at[idx, 'Vol Label']     = label
    tech_df.at[idx, 'Vol Inference'] = inference
    s_label, s_detail = get_sector_trend(row['Sector'])
    tech_df.at[idx, 'Sector Trend']  = s_label
    tech_df.at[idx, 'Sector Detail'] = s_detail


# ══════════════════════════════════════════════════════════════════════════════
# PRINT REPORT
# ══════════════════════════════════════════════════════════════════════════════
today = datetime.now().strftime('%d %B %Y')
print(f"{'='*78}")
print(f"  AI STOCK SCREENER — WEEKLY REPORT")
print(f"  {today}")
print(f"{'='*78}")
print(f"\n  SecRank  = rank vs same sector  |  CapRank = rank vs same sector + same cap")
print(f"  Week Vol = this week avg vs prior 15-day avg")
print(f"  Break Vol= this week avg vs consolidation period avg (breakout signal)")

# ── TIER 1 ────────────────────────────────────────────────────────────────────
tier1 = tech_df[tech_df['Tier'].str.startswith('TIER 1')].copy()
tier1['_cap_order'] = tier1['Cap Category'].apply(
    lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier1 = tier1.sort_values(['_cap_order', 'Fund Score'],
                          ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 1 — ACT NOW  ({len(tier1)} stocks)")
print(f"  Fundamentally strong + Technical setup confirmed")
print(f"{'─'*78}")

serial = 1
current_cap = None
for _, row in tier1.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_line = ''
    if row['In Consolidation'] and row['Consol Days'] > 0:
        bvol_inf = get_breakout_vol_inference(
            row['Vol 5D Ratio'], row['Breakout Vol'])
        consol_line = (f"\n    Base    : {row['Consol Days']}d "
                      f"({row['Consol Label']}) | "
                      f"{row['Pct to Breakout']:+.1f}% to breakout"
                      f"\n    Break   : {bvol_inf}")

    print(f"""
  #{serial}  {row['Symbol']}  ({row['Sector']})  MCap Rs{row['Market Cap Cr']:,.0f}Cr  Rs{row['Current Price']:.2f}
    Setup   : {row['Best Setup']:10}  Tech {row['Tech Score']:3.0f}/100
    Scores  : Fund {row['Fund Score']}/100  Sector Rank {row['Sector Score']}/10  Cap Rank {row['Cap Score']}/10{consol_line}
    Tech    : RSI {row['RSI']:.0f}  ADX {row['ADX']:.0f}  MACD {row['MACD Hist']:+.2f}  → {get_reason(row)}
    Volume  : Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) → {row['Vol Inference']}
    Sector  : {row['Sector Trend']} | {row['Sector Detail']}""")
    serial += 1

# ── TIER 2A ───────────────────────────────────────────────────────────────────
tier2a = tech_df[tech_df['Tier'] == 'TIER 2 — WATCHLIST'].copy()
tier2a = tier2a.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2A — WATCHLIST  ({len(tier2a)} stocks)")
print(f"  Setup forming — wait for confirmation before entering")
print(f"{'─'*78}")

for _, row in tier2a.iterrows():
    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
    print(f"      Technical : [{row['Best Setup']}|{row['Tech Score']:.0f}]  "
          f"RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
    print(f"      Volume    : Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) "
          f"→ {row['Vol Inference']}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── TIER 2B ───────────────────────────────────────────────────────────────────
tier2b = tech_df[tech_df['Tier'] == 'TIER 2 — BASE BUILDING'].copy()
tier2b = tier2b.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2B — BASE BUILDING  ({len(tier2b)} stocks)")
print(f"  Long consolidation base — watch for volume breakout")
print(f"{'─'*78}")

for _, row in tier2b.iterrows():
    bvol_inf = get_breakout_vol_inference(
        row['Vol 5D Ratio'], row['Breakout Vol'])
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}  "
          f"{row['Consol Days']}d  "
          f"Range:{row['range_pct']:.1f}%  "
          f"ToBreak:{row['Pct to Breakout']:+.1f}%")
    print(f"      Technical : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
    print(f"      Volume    : {bvol_inf}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── TIER 3 ────────────────────────────────────────────────────────────────────
tier3 = tech_df[tech_df['Tier'] == 'TIER 3 — WAITING'].copy()
tier3['_cap_order'] = tier3['Cap Category'].apply(
    lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier3 = tier3.sort_values(['_cap_order', 'Fund Score'],
                          ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 3 — WAITING  ({len(tier3)} stocks)")
print(f"  Good businesses — technical setup not ready yet")
print(f"{'─'*78}")

current_cap = None
for _, row in tier3.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''

    if row['In Consolidation'] and row['Breakout Vol'] > 0:
        vol_line = get_breakout_vol_inference(
            row['Vol 5D Ratio'], row['Breakout Vol'])
    else:
        vol_line = (f"Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) "
                   f"→ {row['Vol Inference']}")

    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
    print(f"      Technical : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
    print(f"      Volume    : {vol_line}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── LONG BASE WATCH ───────────────────────────────────────────────────────────
consol_stocks = tech_df[tech_df['In Consolidation'] == True].copy()
consol_stocks = consol_stocks.sort_values('Consol Days', ascending=False)

print(f"\n{'─'*78}")
print(f"  LONG BASE WATCH  ({len(consol_stocks)} stocks, 300+ days)")
print(f"  Watch for breakout above range high — Break Vol > 1.5x = signal building")
print(f"{'─'*78}")
print(f"  {'Symbol':12} {'Category':14} {'Days':6} {'Range%':7} {'ToBreak':8} "
      f"{'WkVol':7} {'BrkVol':7}  Signal & Status")
print(f"  {'─'*90}")

for _, row in consol_stocks.iterrows():
    pct      = row['Pct to Breakout']
    status   = "NEAR BREAKOUT ⚡" if pct > -5 else \
               "Approaching"      if pct > -15 else "Early stage"
    bvol     = row['Breakout Vol']
    wvol     = row['Vol 5D Ratio']
    bvol_tag = "⚡ BUILDING"  if bvol >= 1.5 else \
               "~ normal"     if bvol >= 0.8 else "→ quiet"

    print(f"  {row['Symbol']:12} "
          f"{row['Cap Category']:14} "
          f"{row['Consol Days']:5}d "
          f"{row['range_pct']:6.1f}% "
          f"{pct:+7.1f}%  "
          f"{wvol:.2f}x  "
          f"{bvol:.2f}x  "
          f"{bvol_tag:12}  [{status}]")

print(f"\n{'─'*78}")
print(f"  L=Large Cap  ML=Mini Large Cap  M=Mid Cap  S=Small Cap")
print(f"  SecRank  = rank vs same sector | CapRank = rank vs same sector + cap")
print(f"  Week Vol = this week avg vs prior 15-day avg")
print(f"  Break Vol= this week avg vs consolidation period avg")
print(f"{'─'*78}")

# ── Save ──────────────────────────────────────────────────────────────────────
tech_df.to_csv('../data/technical_report.csv', index=False)
print("\nSaved: data/technical_report.csv ✅")

  AI STOCK SCREENER — WEEKLY REPORT
  21 March 2026

  SecRank  = rank vs same sector  |  CapRank = rank vs same sector + same cap
  Week Vol = this week avg vs prior 15-day avg
  Break Vol= this week avg vs consolidation period avg (breakout signal)

──────────────────────────────────────────────────────────────────────────────
  TIER 1 — ACT NOW  (4 stocks)
  Fundamentally strong + Technical setup confirmed
──────────────────────────────────────────────────────────────────────────────

  [L] Large Cap
  ────────────────────────────────────────────────────────────────────

  #1  SUNPHARMA  (Healthcare)  MCap Rs432,264Cr  Rs1801.60
    Setup   : Momentum    Tech 100/100
    Scores  : Fund 65.0/100  Sector Rank 10.0/10  Cap Rank 10.0/10
    Base    : 520d (520d (1.5-2 years)) | -2.7% to breakout
    Break   : Week Vol:1.23x | Break Vol:1.29x ~  → Normal volume, no breakout pressure
    Tech    : RSI 61  ADX 40  MACD +5.11  → Uptrend confirmed, RSI 61, ADX 40
    Volume  : Week Vol:1.23x

In [23]:
# ── TASK 3: BEARISH REVERSAL SCORER ──────────────────────────────────────────
def score_bearish_reversal(df):
    """
    5 strict filters for bearish reversal signal.
    Returns (signal: bool, score: int, details: dict)
    Score 5/5 = strong signal, 3-4 = watch, <3 = no signal
    """
    if len(df) < 60:
        return False, 0, {}

    latest  = df.iloc[-1]
    recent  = df.iloc[-20:]   # last 20 days
    prev5   = df.iloc[-5:]    # last 5 days

    close   = df['Close']
    high    = df['High']
    rsi     = df['RSI']
    macd_h  = df['MACD_Hist']
    adx     = df['ADX']
    volume  = df['Volume']

    details = {}
    score   = 0

    # ── Filter 1: Run up > 30% from 52-week low ───────────────────────────────
    low_52w    = close.iloc[-252:].min() if len(close) >= 252 else close.min()
    current    = close.iloc[-1]
    runup_pct  = (current - low_52w) / low_52w * 100
    f1         = runup_pct >= 30
    details['runup_pct']  = round(runup_pct, 1)
    details['filter1_runup'] = f1
    if f1: score += 1

    # ── Filter 2: RSI > 70 for 5 consecutive days ─────────────────────────────
    rsi_last5  = rsi.iloc[-5:]
    f2         = (rsi_last5 > 70).all()
    details['rsi_last5_min'] = round(rsi_last5.min(), 1)
    details['filter2_rsi_extended'] = f2
    if f2: score += 1

    # ── Filter 3: RSI divergence + MACD divergence (BOTH required) ───────────
    # RSI divergence: price making higher high but RSI making lower high
    price_high_recent = close.iloc[-10:].max()
    price_high_prev   = close.iloc[-20:-10].max()
    rsi_high_recent   = rsi.iloc[-10:].max()
    rsi_high_prev     = rsi.iloc[-20:-10].max()
    rsi_div           = (price_high_recent > price_high_prev) and \
                        (rsi_high_recent   < rsi_high_prev)

    # MACD divergence: price higher high but MACD histogram lower high
    macd_high_recent  = macd_h.iloc[-10:].max()
    macd_high_prev    = macd_h.iloc[-20:-10].max()
    macd_div          = (price_high_recent > price_high_prev) and \
                        (macd_high_recent  < macd_high_prev)

    f3 = rsi_div and macd_div
    details['rsi_divergence']  = rsi_div
    details['macd_divergence'] = macd_div
    details['filter3_both_div'] = f3
    if f3: score += 1

    # ── Filter 4: ADX was > 30, now declining ────────────────────────────────
    adx_peak   = adx.iloc[-15:-5].max()   # peak in last 15-5 days
    adx_now    = adx.iloc[-1]
    f4         = (adx_peak > 30) and (adx_now < adx_peak * 0.85)  # declined 15%+
    details['adx_peak']          = round(adx_peak, 1)
    details['adx_now']           = round(adx_now,  1)
    details['filter4_adx_fade']  = f4
    if f4: score += 1

    # ── Filter 5: Volume declining at price highs ────────────────────────────
    # Compare volume on up days recently vs earlier
    recent_up_vol = volume.iloc[-10:][close.iloc[-10:].diff() > 0].mean()
    prev_up_vol   = volume.iloc[-20:-10][close.iloc[-20:-10].diff() > 0].mean()
    vol_declining = (recent_up_vol < prev_up_vol * 0.85) if prev_up_vol > 0 else False
    f5            = vol_declining
    details['vol_ratio_up_days'] = round(recent_up_vol / prev_up_vol, 2) \
                                   if prev_up_vol > 0 else 0
    details['filter5_vol_decline'] = f5
    if f5: score += 1

    # ── Signal threshold ──────────────────────────────────────────────────────
    signal = score >= 3

    return signal, score, details


# ── Test on all 35 stocks ─────────────────────────────────────────────────────
print("Scanning for bearish reversal signals...")
print(f"\n{'Symbol':12} {'Score':6} {'Runup%':8} {'RSI>70×5':9} "
      f"{'RSI Div':8} {'MACD Div':9} {'ADX Fade':9} {'Vol↓':6}  Signal")
print("─" * 90)

bearish_results = {}
for symbol, df in indicator_data.items():
    if symbol not in [s for s in tech_df['Symbol']]:
        continue
    signal, score, details = score_bearish_reversal(df)
    bearish_results[symbol] = {
        'signal': signal, 'score': score, 'details': details
    }
    if score > 0:  # only show stocks with at least 1 filter hit
        print(f"  {symbol:12} {score}/5    "
              f"{details.get('runup_pct', 0):+6.1f}%   "
              f"{'✓' if details.get('filter2_rsi_extended') else '✗':8}  "
              f"{'✓' if details.get('rsi_divergence') else '✗':7}  "
              f"{'✓' if details.get('macd_divergence') else '✗':8}  "
              f"{'✓' if details.get('filter4_adx_fade') else '✗':8}  "
              f"{'✓' if details.get('filter5_vol_decline') else '✗':5}  "
              f"{'⚠ SIGNAL' if signal else ''}")

triggered = [s for s, v in bearish_results.items() if v['signal']]
print(f"\nStocks with signal (3+/5): {triggered}")

Scanning for bearish reversal signals...

Symbol       Score  Runup%   RSI>70×5  RSI Div  MACD Div  ADX Fade  Vol↓    Signal
──────────────────────────────────────────────────────────────────────────────────────────
  RELIANCE     1/5     +18.9%   ✗         ✗        ✗         ✓         ✗      
  WIPRO        1/5      +1.1%   ✗         ✗        ✗         ✓         ✗      
  PERSISTENT   1/5      +6.0%   ✗         ✗        ✗         ✗         ✓      
  LTTS         1/5     +10.7%   ✗         ✗        ✗         ✓         ✗      
  MUTHOOTFIN   1/5     +67.3%   ✗         ✗        ✗         ✗         ✗      
  GARFIBRES    1/5      +1.4%   ✗         ✗        ✗         ✓         ✗      
  SUPRAJIT     1/5     +11.6%   ✗         ✗        ✗         ✗         ✓      
  HAL          1/5     +16.6%   ✗         ✗        ✗         ✓         ✗      
  BEL          1/5     +63.3%   ✗         ✗        ✗         ✗         ✗      
  PARAS        1/5     +44.6%   ✗         ✗        ✗         ✗         ✗ 

In [24]:
# ── Debug ABSLAMC and MUTHOOTFIN ──────────────────────────────────────────────
for symbol in ['ABSLAMC', 'MUTHOOTFIN', 'BEL', 'TITAN', 'IPCALAB']:
    if symbol not in indicator_data:
        continue
    signal, score, details = score_bearish_reversal(indicator_data[symbol])
    print(f"\n{symbol}  Score:{score}/5")
    print(f"  Runup from 52w low : {details.get('runup_pct', 0):+.1f}%  "
          f"  Filter1: {details.get('filter1_runup')}")
    print(f"  RSI last 5d min    : {details.get('rsi_last5_min', 0):.1f}  "
          f"  Filter2: {details.get('filter2_rsi_extended')}")
    print(f"  RSI divergence     : {details.get('rsi_divergence')}  "
          f"  MACD divergence: {details.get('macd_divergence')}  "
          f"  Filter3: {details.get('filter3_both_div')}")
    print(f"  ADX peak:{details.get('adx_peak',0):.1f}  "
          f"  ADX now:{details.get('adx_now',0):.1f}  "
          f"  Filter4: {details.get('filter4_adx_fade')}")
    print(f"  Vol ratio up days  : {details.get('vol_ratio_up_days', 0):.2f}  "
          f"  Filter5: {details.get('filter5_vol_decline')}")


ABSLAMC  Score:3/5
  Runup from 52w low : +62.1%    Filter1: True
  RSI last 5d min    : 44.2    Filter2: False
  RSI divergence     : True    MACD divergence: True    Filter3: True
  ADX peak:56.7    ADX now:35.3    Filter4: True
  Vol ratio up days  : 4.63    Filter5: False

MUTHOOTFIN  Score:1/5
  Runup from 52w low : +67.3%    Filter1: True
  RSI last 5d min    : 27.8    Filter2: False
  RSI divergence     : False    MACD divergence: False    Filter3: False
  ADX peak:28.9    ADX now:40.1    Filter4: False
  Vol ratio up days  : 1.63    Filter5: False

BEL  Score:1/5
  Runup from 52w low : +63.3%    Filter1: True
  RSI last 5d min    : 40.3    Filter2: False
  RSI divergence     : False    MACD divergence: False    Filter3: False
  ADX peak:33.4    ADX now:28.7    Filter4: False
  Vol ratio up days  : 2.47    Filter5: False

TITAN  Score:1/5
  Runup from 52w low : +36.8%    Filter1: True
  RSI last 5d min    : 31.1    Filter2: False
  RSI divergence     : False    MACD divergence:

In [27]:
# ── SECTOR TREND SUMMARY (add before Tier 1) ──────────────────────────────────
def print_sector_summary():
    sectors_in_use = sorted(tech_df['Sector'].unique())
    print(f"\n{'─'*78}")
    print(f"  SECTOR TREND SUMMARY")
    print(f"{'─'*78}")
    print(f"  {'Sector':40} {'Trend':28} Detail")
    print(f"  {'─'*76}")
    for sector in sectors_in_use:
        label, detail = get_sector_trend(sector)
        print(f"  {sector:40} {label:28} {detail}")


# ── BEARISH REVERSAL SECTION (add after Long Base Watch) ──────────────────────
def print_bearish_reversal_section():
    print(f"\n{'─'*78}")
    print(f"  ⚠  BEARISH REVERSAL WATCH")
    print(f"  Stocks showing distribution signals — consider partial profit booking (25-30%)")
    print(f"  Signal fires when 3 or more of 5 filters trigger")
    print(f"{'─'*78}")
    print(f"  Filters: (1)Runup>30%  (2)RSI>70×5d  (3)RSI+MACD divergence")
    print(f"           (4)ADX fading  (5)Volume declining on up days")
    print(f"{'─'*78}")

    triggered_stocks = [(s, v) for s, v in bearish_results.items() if v['signal']]

    if not triggered_stocks:
        print(f"\n  No stocks triggered this week. Market is clean.")
        return

    for symbol, result in triggered_stocks:
        score   = result['score']
        details = result['details']
        row     = tech_df[tech_df['Symbol'] == symbol].iloc[0]

        # Which filters fired
        f1 = '✓' if details.get('filter1_runup')          else '✗'
        f2 = '✓' if details.get('filter2_rsi_extended')   else '✗'
        f3 = '✓' if details.get('filter3_both_div')       else '✗'
        f4 = '✓' if details.get('filter4_adx_fade')       else '✗'
        f5 = '✓' if details.get('filter5_vol_decline')    else '✗'

        # Inference
        if score == 5:
            inference = "Very strong distribution signal → book 25-30% immediately"
        elif score == 4:
            inference = "Strong signal → start scaling out 20-25%"
        else:
            inference = "Early warning → monitor closely, avoid adding"

        print(f"""
  ⚠  {symbol}  ({row['Sector']})  Rs{row['Current Price']:.2f}  Score:{score}/5
     Runup from 52w low : {details.get('runup_pct', 0):+.1f}%
     Filters fired      : (1){f1} Runup  (2){f2} RSI>70×5d  (3){f3} Divergence  (4){f4} ADX fade  (5){f5} Vol↓
     RSI divergence     : {details.get('rsi_divergence')}  |  MACD divergence: {details.get('macd_divergence')}
     ADX peak→now       : {details.get('adx_peak', 0):.1f} → {details.get('adx_now', 0):.1f}  (fading trend strength)
     Sector             : {row['Sector Trend']} | {row['Sector Detail']}
     Action             : {inference}""")

    print(f"\n{'─'*78}")
    print(f"  Note: This is NOT a sell signal. Quality stocks may continue higher.")
    print(f"  Use to protect partial gains. Re-enter if pullback holds key levels.")
    print(f"{'─'*78}")

In [28]:
# In your complete report cell, add these two calls:

# -- After the legend lines and before Tier 1 print block --
print_sector_summary()

# -- After the Long Base Watch block and before the footer --
print_bearish_reversal_section()


──────────────────────────────────────────────────────────────────────────────
  SECTOR TREND SUMMARY
──────────────────────────────────────────────────────────────────────────────
  Sector                                   Trend                        Detail
  ────────────────────────────────────────────────────────────────────────────
  Automobile and Auto Components           Strong Downtrend ↓↓          RSI 37.2 | ADX 40.3 | MACD ▼ | EMA mixed | Below EMA200
  Capital Goods                            Strong Downtrend ↓↓          RSI 36.0 | ADX 40.7 | MACD ▼ | EMA mixed | Below EMA200
  Chemicals                                Strong Downtrend ↓↓          RSI 34.4 | ADX 42.3 | MACD ▼ | EMA mixed | Below EMA200
  Consumer Durables                        Strong Downtrend ↓↓          RSI 36.4 | ADX 46.3 | MACD ▼ | EMA ↓↓↓ | Below EMA200
  Consumer Services                        Strong Downtrend ↓↓          RSI 34.4 | ADX 42.3 | MACD ▼ | EMA mixed | Below EMA200
  Financial Services  

In [29]:
# ── SECTOR TREND SUMMARY — ALL MAPPED SECTORS ─────────────────────────────────
print(f"\n{'─'*78}")
print(f"  SECTOR TREND SUMMARY  (full market)")
print(f"{'─'*78}")
print(f"  {'Sector':42} {'Trend':28} Detail")
print(f"  {'─'*76}")

# Get unique sectors from map, skip duplicates that share same index
seen_labels = {}
for sector in sorted(SECTOR_INDEX_MAP.keys()):
    ticker = SECTOR_INDEX_MAP[sector]
    if ticker not in sector_price:
        print(f"  {sector:42} {'No data':28} —")
        continue
    label, detail = get_sector_trend(sector)
    print(f"  {sector:42} {label:28} {detail}")

print(f"{'─'*78}")


──────────────────────────────────────────────────────────────────────────────
  SECTOR TREND SUMMARY  (full market)
──────────────────────────────────────────────────────────────────────────────
  Sector                                     Trend                        Detail
  ────────────────────────────────────────────────────────────────────────────
  Agriculture                                No data                      —
  Automobile and Auto Components             No data                      —
  Banking                                    No data                      —
  Capital Goods                              No data                      —
  Cement                                     No data                      —
  Chemicals                                  No data                      —
  Construction                               No data                      —
  Consumer Durables                          No data                      —
  Consumer Services                

In [30]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPLETE WEEKLY REPORT — FINAL VERSION
# ══════════════════════════════════════════════════════════════════════════════
from datetime import datetime

CAP_ORDER = ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']

# ── Helper functions ──────────────────────────────────────────────────────────
def get_reason(row):
    rsi    = float(row['RSI'])
    adx    = float(row['ADX'])
    setup  = str(row['Best Setup'])
    hist   = float(row['MACD Hist'])
    di_p   = float(row.get('DI+', 0))
    di_m   = float(row.get('DI-', 0))
    ema50  = float(row.get('EMA50',  0))
    ema200 = float(row.get('EMA200', 0))
    price  = float(row.get('Current Price', 0))

    uptrend   = (adx > 25 and di_p > di_m and price > ema50 and price > ema200)
    bounce    = (price > ema50 and price < ema200)
    downtrend = (price < ema200) or (adx > 25 and di_m > di_p)

    if setup == 'Momentum':
        if uptrend:                        return f"Uptrend confirmed, RSI {rsi:.0f}, ADX {adx:.0f}"
        elif bounce:                       return f"Short bounce, below EMA200"
        elif price < ema200 and rsi > 50:  return f"RSI {rsi:.0f} bounce in downtrend"
        elif rsi >= 50:                    return f"Momentum building, RSI {rsi:.0f}"
        return f"Momentum setup, RSI {rsi:.0f}"
    elif setup == 'Reversal':
        if rsi < 35 and hist > 0:          return f"Oversold RSI {rsi:.0f}, MACD turning up"
        elif rsi < 35:                     return f"Oversold RSI {rsi:.0f}, watch for turn"
        elif hist > 0 and downtrend:       return f"Downtrend weakening, MACD improving"
        return f"Divergence forming, RSI {rsi:.0f}"
    else:
        if downtrend and adx > 40:         return f"Strong downtrend ADX {adx:.0f}, wait"
        elif downtrend and adx > 25:       return f"Downtrend ADX {adx:.0f}, no setup yet"
        elif bounce:                       return f"Bounce in downtrend, below EMA200"
        elif rsi < 25:                     return f"Deeply oversold RSI {rsi:.0f}, too early"
        elif rsi < 40:                     return f"Oversold RSI {rsi:.0f}, watch for turn"
        elif rsi > 60 and not uptrend:     return f"RSI {rsi:.0f} bounce, below EMA200"
        else:                              return f"RSI {rsi:.0f}, setup not confirmed"


def get_volume_inference(row):
    vol   = float(row['Vol 5D Ratio'])
    setup = str(row['Best Setup'])
    rsi   = float(row['RSI'])
    adx   = float(row['ADX'])

    if   vol >= 3.0: vol_label = "Extremely High"
    elif vol >= 2.0: vol_label = "Very High"
    elif vol >= 1.5: vol_label = "High"
    elif vol >= 1.0: vol_label = "Normal"
    elif vol >= 0.7: vol_label = "Low"
    else:            vol_label = "Very Low"

    if setup == 'Momentum':
        if   vol >= 3.0: return vol_label, "Exceptional participation → strong conviction"
        elif vol >= 2.0: return vol_label, "Strong participation → trend likely to continue"
        elif vol >= 1.5: return vol_label, "Good volume support → momentum confirmed"
        elif vol >= 1.0: return vol_label, "Average volume → monitor for increase"
        elif vol >= 0.7: return vol_label, "Weak volume → momentum may fade, wait"
        else:            return vol_label, "Very low volume → no conviction, caution"
    elif setup == 'Reversal':
        if   vol >= 3.0: return vol_label, "Very high volume on reversal → strong capitulation signal"
        elif vol >= 2.0: return vol_label, "High volume → could be capitulation or distribution"
        elif vol >= 1.5: return vol_label, "Elevated volume on reversal → watch direction carefully"
        elif vol >= 1.0: return vol_label, "Neutral volume → wait for low-volume base to form"
        elif vol >= 0.7: return vol_label, "Selling pressure easing → watch for turn"
        else:            return vol_label, "Sellers exhausted → reversal likely"
    else:
        if   vol >= 3.0 and adx > 40: return vol_label, "Very high volume downtrend → capitulation possible"
        elif vol >= 3.0:              return vol_label, "Exceptional volume → major event this week, watch"
        elif vol >= 2.0 and adx > 40: return vol_label, "High volume downtrend → stay away"
        elif vol >= 2.0 and rsi < 35: return vol_label, "High volume selloff → possible capitulation"
        elif vol >= 2.0:              return vol_label, "Above avg volume but no setup → watch closely"
        elif vol >= 1.5 and adx > 40: return vol_label, "Strong downtrend → no entry signal"
        elif vol >= 1.5:              return vol_label, "Above avg volume but no setup → watch"
        elif vol >= 1.0:              return vol_label, "Normal activity → no setup forming yet"
        elif vol >= 0.7:              return vol_label, "Low activity → no setup forming yet"
        else:                         return vol_label, "Very low volume → no interest, wait"


def get_breakout_vol_inference(week_vol, break_vol):
    if   break_vol >= 2.0: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ⚡⚡ → Strong breakout volume building"
    elif break_vol >= 1.5: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ⚡  → Volume building toward breakout"
    elif break_vol >= 1.0: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ~  → Normal volume, no breakout pressure"
    elif break_vol >= 0.7: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x    → Quiet, below consolidation avg"
    else:                  return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x    → Very quiet, no breakout interest"


def calculate_sector_indicators(df):
    data = df.copy()
    data['EMA20']  = data['Close'].ewm(span=20,  adjust=False).mean()
    data['EMA50']  = data['Close'].ewm(span=50,  adjust=False).mean()
    data['EMA200'] = data['Close'].ewm(span=200, adjust=False).mean()
    delta    = data['Close'].diff()
    gain     = delta.where(delta > 0, 0)
    loss     = -delta.where(delta < 0, 0)
    avg_gain = gain.ewm(span=14, adjust=False).mean()
    avg_loss = loss.ewm(span=14, adjust=False).mean()
    rs       = avg_gain / (avg_loss + 1e-9)
    data['RSI'] = 100 - (100 / (1 + rs))
    ema12          = data['Close'].ewm(span=12, adjust=False).mean()
    ema26          = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD']   = ema12 - ema26
    data['Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()
    data['MACD_Hist'] = data['MACD'] - data['Signal']
    high     = data['High']
    low      = data['Low']
    close    = data['Close']
    tr1      = high - low
    tr2      = (high - close.shift()).abs()
    tr3      = (low  - close.shift()).abs()
    tr       = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    dm_plus  = high.diff()
    dm_minus = -low.diff()
    dm_plus  = dm_plus.where((dm_plus > dm_minus) & (dm_plus > 0), 0)
    dm_minus = dm_minus.where((dm_minus > dm_plus) & (dm_minus > 0), 0)
    atr      = tr.ewm(span=14, adjust=False).mean()
    di_plus  = 100 * dm_plus.ewm(span=14, adjust=False).mean() / (atr + 1e-9)
    di_minus = 100 * dm_minus.ewm(span=14, adjust=False).mean() / (atr + 1e-9)
    dx       = 100 * (di_plus - di_minus).abs() / (di_plus + di_minus + 1e-9)
    data['ADX']      = dx.ewm(span=14, adjust=False).mean()
    data['DI_Plus']  = di_plus
    data['DI_Minus'] = di_minus
    return data


def get_sector_trend(sector):
    if sector not in sector_price:
        return "No Data", "N/A"
    df     = calculate_sector_indicators(sector_price[sector])
    latest = df.iloc[-1]
    close    = latest['Close']
    rsi      = round(latest['RSI'],   1)
    adx      = round(latest['ADX'],   1)
    ema20    = latest['EMA20']
    ema50    = latest['EMA50']
    ema200   = latest['EMA200']
    di_plus  = latest['DI_Plus']
    di_minus = latest['DI_Minus']
    macd     = latest['MACD']
    signal   = latest['Signal']
    macd_h   = latest['MACD_Hist']
    above_ema200 = close > ema200
    above_ema50  = close > ema50
    ema_aligned  = ema20 > ema50 > ema200
    ema_bearish  = ema20 < ema50 < ema200
    uptrend      = di_plus > di_minus
    macd_bull    = macd > signal and macd_h > 0
    macd_bear    = macd < signal and macd_h < 0
    macd_turning = macd_h > 0 and macd < signal
    bull_score = sum([above_ema200, above_ema50, ema_aligned,
                      uptrend, macd_bull, rsi > 55])
    bear_score = sum([not above_ema200, not above_ema50, ema_bearish,
                      not uptrend, macd_bear, rsi < 45])
    if   bull_score >= 5 and adx > 25:  label = "Strong Uptrend ↑↑"
    elif bull_score >= 4 and adx > 20:  label = "Uptrend ↑"
    elif bull_score >= 3 and adx <= 20: label = "Weak Uptrend →↑"
    elif bull_score == bear_score or adx < 15: label = "Sideways →"
    elif bear_score >= 5 and adx > 25:  label = "Strong Downtrend ↓↓"
    elif bear_score >= 4 and adx > 20:  label = "Downtrend ↓"
    elif bear_score >= 3 and adx <= 20: label = "Weak Downtrend →↓"
    else: label = "Sideways →"
    if macd_turning and bear_score >= 3:
        label += " (MACD turning)"
    ema_str  = '↑↑↑' if ema_aligned else '↓↓↓' if ema_bearish else 'mixed'
    macd_str = '▲' if macd_bull else '▼' if macd_bear else '~'
    detail   = (f"RSI {rsi} | ADX {adx} | MACD {macd_str} | "
                f"EMA {ema_str} | {'Above' if above_ema200 else 'Below'} EMA200")
    return label, detail


# ── Rebuild Vol Label, Vol Inference, Sector Trend ────────────────────────────
for idx, row in tech_df.iterrows():
    label, inference = get_volume_inference(row)
    tech_df.at[idx, 'Vol Label']     = label
    tech_df.at[idx, 'Vol Inference'] = inference
    s_label, s_detail = get_sector_trend(row['Sector'])
    tech_df.at[idx, 'Sector Trend']  = s_label
    tech_df.at[idx, 'Sector Detail'] = s_detail

# ── Re-fetch sector_price if needed ───────────────────────────────────────────
if 'sector_price' not in dir() or len(sector_price) == 0:
    print("Re-fetching sector index data...")
    SECTOR_INDEX_MAP = {
        'Information Technology'             : '^CNXIT',
        'Healthcare'                         : '^CNXPHARMA',
        'Financial Services'                 : 'FINIETF.NS',
        'Capital Goods'                      : '^CNXINFRA',
        'Chemicals'                          : 'MID150BEES.NS',
        'Consumer Durables'                  : '^CNXCONSUM',
        'Oil, Gas & Consumable Fuels'        : '^CNXENERGY',
        'Automobile and Auto Components'     : '^CNXAUTO',
        'Textiles'                           : 'MID150BEES.NS',
        'Consumer Services'                  : 'MID150BEES.NS',
        'Banking'                            : '^NSEBANK',
        'Private Banking'                    : 'PVTBANIETF.NS',
        'PSU Banking'                        : '^CNXPSUBANK',
        'Fast Moving Consumer Goods'         : '^CNXFMCG',
        'Metals & Mining'                    : '^CNXMETAL',
        'Realty'                             : '^CNXREALTY',
        'Media Entertainment & Publication'  : '^CNXMEDIA',
        'Pharma'                             : '^CNXPHARMA',
        'Services'                           : '^CNXSERVICE',
        'Construction'                       : '^CNXINFRA',
        'Power'                              : '^CNXENERGY',
        'Defence'                            : 'MID150BEES.NS',
        'Telecommunication'                  : 'MID150BEES.NS',
        'Diversified'                        : 'MID150BEES.NS',
        'Forest Materials'                   : 'MID150BEES.NS',
        'Agriculture'                        : 'MID150BEES.NS',
        'Cement'                             : '^CNXCMDT',
    }
    sector_price = {}
    ticker_data  = {}
    for ticker in list(set(SECTOR_INDEX_MAP.values())):
        try:
            df = yf.Ticker(ticker).history(period='1y')
            if len(df) > 50:
                ticker_data[ticker] = df
        except:
            pass
    fallback = 'MID150BEES.NS'
    for sector, ticker in SECTOR_INDEX_MAP.items():
        if ticker in ticker_data:
            sector_price[sector] = ticker_data[ticker]
        elif fallback in ticker_data:
            sector_price[sector] = ticker_data[fallback]
    print(f"Fetched: {len(sector_price)}/27 sectors")

# ── Bearish reversal scan ─────────────────────────────────────────────────────
def score_bearish_reversal(df):
    if len(df) < 60:
        return False, 0, {}
    close   = df['Close']
    rsi     = df['RSI']
    macd_h  = df['MACD_Hist']
    adx     = df['ADX']
    volume  = df['Volume']
    details = {}
    score   = 0
    low_52w   = close.iloc[-252:].min() if len(close) >= 252 else close.min()
    current   = close.iloc[-1]
    runup_pct = (current - low_52w) / low_52w * 100
    f1        = runup_pct >= 30
    details['runup_pct']         = round(runup_pct, 1)
    details['filter1_runup']     = f1
    if f1: score += 1
    rsi_last5 = rsi.iloc[-5:]
    f2        = (rsi_last5 > 70).all()
    details['rsi_last5_min']          = round(rsi_last5.min(), 1)
    details['filter2_rsi_extended']   = f2
    if f2: score += 1
    price_high_recent = close.iloc[-10:].max()
    price_high_prev   = close.iloc[-20:-10].max()
    rsi_high_recent   = rsi.iloc[-10:].max()
    rsi_high_prev     = rsi.iloc[-20:-10].max()
    rsi_div           = (price_high_recent > price_high_prev) and \
                        (rsi_high_recent   < rsi_high_prev)
    macd_high_recent  = macd_h.iloc[-10:].max()
    macd_high_prev    = macd_h.iloc[-20:-10].max()
    macd_div          = (price_high_recent > price_high_prev) and \
                        (macd_high_recent  < macd_high_prev)
    f3 = rsi_div and macd_div
    details['rsi_divergence']    = rsi_div
    details['macd_divergence']   = macd_div
    details['filter3_both_div']  = f3
    if f3: score += 1
    adx_peak  = adx.iloc[-15:-5].max()
    adx_now   = adx.iloc[-1]
    f4        = (adx_peak > 30) and (adx_now < adx_peak * 0.85)
    details['adx_peak']          = round(adx_peak, 1)
    details['adx_now']           = round(adx_now,  1)
    details['filter4_adx_fade']  = f4
    if f4: score += 1
    recent_up_vol = volume.iloc[-10:][close.iloc[-10:].diff() > 0].mean()
    prev_up_vol   = volume.iloc[-20:-10][close.iloc[-20:-10].diff() > 0].mean()
    vol_declining = (recent_up_vol < prev_up_vol * 0.85) if prev_up_vol > 0 else False
    f5            = vol_declining
    details['vol_ratio_up_days']     = round(recent_up_vol / prev_up_vol, 2) \
                                       if prev_up_vol > 0 else 0
    details['filter5_vol_decline']   = f5
    if f5: score += 1
    signal = score >= 3
    return signal, score, details

bearish_results = {}
for symbol, df in indicator_data.items():
    if symbol not in tech_df['Symbol'].values:
        continue
    signal, score, details = score_bearish_reversal(df)
    bearish_results[symbol] = {'signal': signal, 'score': score, 'details': details}

# ── 5D vs 15D volume ─────────────────────────────────────────────────────────
vol_5d_data = {}
for symbol, df in price_data.items():
    try:
        if len(df) < 20:
            vol_5d_data[symbol] = 1.0
            continue
        vol_5d_avg      = df['Volume'].iloc[-5:].mean()
        vol_prior15_avg = df['Volume'].iloc[-20:-5].mean()
        vol_5d_data[symbol] = round(vol_5d_avg / vol_prior15_avg
                                    if vol_prior15_avg > 0 else 1.0, 2)
    except:
        vol_5d_data[symbol] = 1.0

tech_df['Vol 5D Ratio'] = tech_df['Symbol'].map(vol_5d_data)

# ── Breakout volume ───────────────────────────────────────────────────────────
breakout_vol_data = {}
for _, crow in consol_info_df.iterrows():
    symbol      = crow['Symbol']
    consol_days = int(crow['consol_days'])
    if symbol not in price_data:
        continue
    df = price_data[symbol].copy()
    if len(df) < consol_days + 5:
        breakout_vol_data[symbol] = 1.0
        continue
    week_vol   = df['Volume'].iloc[-5:].mean()
    consol_avg = df['Volume'].iloc[-consol_days:-5].mean()
    breakout_vol_data[symbol] = round(week_vol / consol_avg
                                      if consol_avg > 0 else 1.0, 2)

tech_df['Breakout Vol'] = tech_df['Symbol'].map(breakout_vol_data).fillna(0)

# ── Merge range_pct ───────────────────────────────────────────────────────────
tech_df = tech_df.merge(
    consol_info_df[['Symbol', 'range_pct']],
    on='Symbol', how='left', suffixes=('', '_new')
)
if 'range_pct_new' in tech_df.columns:
    tech_df['range_pct'] = tech_df['range_pct_new'].fillna(
        tech_df.get('range_pct', 0))
    tech_df.drop(columns=['range_pct_new'], inplace=True)
tech_df['range_pct'] = tech_df['range_pct'].fillna(0)

# ── Rebuild Vol + Sector columns ──────────────────────────────────────────────
for idx, row in tech_df.iterrows():
    label, inference = get_volume_inference(row)
    tech_df.at[idx, 'Vol Label']     = label
    tech_df.at[idx, 'Vol Inference'] = inference
    s_label, s_detail = get_sector_trend(row['Sector'])
    tech_df.at[idx, 'Sector Trend']  = s_label
    tech_df.at[idx, 'Sector Detail'] = s_detail

# ══════════════════════════════════════════════════════════════════════════════
# PRINT REPORT
# ══════════════════════════════════════════════════════════════════════════════
today = datetime.now().strftime('%d %B %Y')
print(f"{'='*78}")
print(f"  AI STOCK SCREENER — WEEKLY REPORT")
print(f"  {today}")
print(f"{'='*78}")
print(f"\n  SecRank  = rank vs same sector  |  CapRank = rank vs same sector + same cap")
print(f"  Week Vol = this week avg vs prior 15-day avg")
print(f"  Break Vol= this week avg vs consolidation period avg (breakout signal)")

# ── SECTOR TREND SUMMARY ──────────────────────────────────────────────────────
print(f"\n{'─'*78}")
print(f"  SECTOR TREND SUMMARY  (full market)")
print(f"{'─'*78}")
print(f"  {'Sector':42} {'Trend':28} Detail")
print(f"  {'─'*76}")
for sector in sorted(SECTOR_INDEX_MAP.keys()):
    if sector not in sector_price:
        print(f"  {sector:42} {'No data':28} —")
        continue
    label, detail = get_sector_trend(sector)
    print(f"  {sector:42} {label:28} {detail}")
print(f"{'─'*78}")

# ── TIER 1 ────────────────────────────────────────────────────────────────────
tier1 = tech_df[tech_df['Tier'].str.startswith('TIER 1')].copy()
tier1['_cap_order'] = tier1['Cap Category'].apply(
    lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier1 = tier1.sort_values(['_cap_order', 'Fund Score'],
                          ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 1 — ACT NOW  ({len(tier1)} stocks)")
print(f"  Fundamentally strong + Technical setup confirmed")
print(f"{'─'*78}")

serial = 1
current_cap = None
for _, row in tier1.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_line = ''
    if row['In Consolidation'] and row['Consol Days'] > 0:
        bvol_inf = get_breakout_vol_inference(
            row['Vol 5D Ratio'], row['Breakout Vol'])
        consol_line = (f"\n    Base    : {row['Consol Days']}d "
                      f"({row['Consol Label']}) | "
                      f"{row['Pct to Breakout']:+.1f}% to breakout"
                      f"\n    Break   : {bvol_inf}")

    print(f"""
  #{serial}  {row['Symbol']}  ({row['Sector']})  MCap Rs{row['Market Cap Cr']:,.0f}Cr  Rs{row['Current Price']:.2f}
    Setup   : {row['Best Setup']:10}  Tech {row['Tech Score']:3.0f}/100
    Scores  : Fund {row['Fund Score']}/100  Sector Rank {row['Sector Score']}/10  Cap Rank {row['Cap Score']}/10{consol_line}
    Tech    : RSI {row['RSI']:.0f}  ADX {row['ADX']:.0f}  MACD {row['MACD Hist']:+.2f}  → {get_reason(row)}
    Volume  : Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) → {row['Vol Inference']}
    Sector  : {row['Sector Trend']} | {row['Sector Detail']}""")
    serial += 1

# ── TIER 2A ───────────────────────────────────────────────────────────────────
tier2a = tech_df[tech_df['Tier'] == 'TIER 2 — WATCHLIST'].copy()
tier2a = tier2a.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2A — WATCHLIST  ({len(tier2a)} stocks)")
print(f"  Setup forming — wait for confirmation before entering")
print(f"{'─'*78}")

for _, row in tier2a.iterrows():
    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
    print(f"      Technical : [{row['Best Setup']}|{row['Tech Score']:.0f}]  "
          f"RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
    print(f"      Volume    : Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) "
          f"→ {row['Vol Inference']}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── TIER 2B ───────────────────────────────────────────────────────────────────
tier2b = tech_df[tech_df['Tier'] == 'TIER 2 — BASE BUILDING'].copy()
tier2b = tier2b.sort_values('Fund Score', ascending=False).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 2B — BASE BUILDING  ({len(tier2b)} stocks)")
print(f"  Long consolidation base — watch for volume breakout")
print(f"{'─'*78}")

for _, row in tier2b.iterrows():
    bvol_inf = get_breakout_vol_inference(
        row['Vol 5D Ratio'], row['Breakout Vol'])
    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}  "
          f"{row['Consol Days']}d  "
          f"Range:{row['range_pct']:.1f}%  "
          f"ToBreak:{row['Pct to Breakout']:+.1f}%")
    print(f"      Technical : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
    print(f"      Volume    : {bvol_inf}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── TIER 3 ────────────────────────────────────────────────────────────────────
tier3 = tech_df[tech_df['Tier'] == 'TIER 3 — WAITING'].copy()
tier3['_cap_order'] = tier3['Cap Category'].apply(
    lambda x: CAP_ORDER.index(x) if x in CAP_ORDER else 99)
tier3 = tier3.sort_values(['_cap_order', 'Fund Score'],
                          ascending=[True, False]).reset_index(drop=True)

print(f"\n{'─'*78}")
print(f"  TIER 3 — WAITING  ({len(tier3)} stocks)")
print(f"  Good businesses — technical setup not ready yet")
print(f"{'─'*78}")

current_cap = None
for _, row in tier3.iterrows():
    cap = row['Cap Category']
    if cap != current_cap:
        current_cap = cap
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}[cap]
        print(f"\n  [{cap_short}] {cap}")
        print(f"  {'─'*68}")

    consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''

    if row['In Consolidation'] and row['Breakout Vol'] > 0:
        vol_line = get_breakout_vol_inference(
            row['Vol 5D Ratio'], row['Breakout Vol'])
    else:
        vol_line = (f"Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) "
                   f"→ {row['Vol Inference']}")

    print(f"\n  #{serial}  {row['Symbol']:10}  "
          f"Rs{row['Current Price']:.2f}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
    print(f"      Technical : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
    print(f"      Volume    : {vol_line}")
    print(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
    serial += 1

# ── LONG BASE WATCH ───────────────────────────────────────────────────────────
consol_stocks = tech_df[tech_df['In Consolidation'] == True].copy()
consol_stocks = consol_stocks.sort_values('Consol Days', ascending=False)

print(f"\n{'─'*78}")
print(f"  LONG BASE WATCH  ({len(consol_stocks)} stocks, 300+ days)")
print(f"  Watch for breakout above range high — Break Vol > 1.5x = signal building")
print(f"{'─'*78}")
print(f"  {'Symbol':12} {'Category':14} {'Days':6} {'Range%':7} {'ToBreak':8} "
      f"{'WkVol':7} {'BrkVol':7}  Signal & Status")
print(f"  {'─'*90}")

for _, row in consol_stocks.iterrows():
    pct      = row['Pct to Breakout']
    status   = "NEAR BREAKOUT ⚡" if pct > -5 else \
               "Approaching"      if pct > -15 else "Early stage"
    bvol     = row['Breakout Vol']
    wvol     = row['Vol 5D Ratio']
    bvol_tag = "⚡ BUILDING"  if bvol >= 1.5 else \
               "~ normal"     if bvol >= 0.8 else "→ quiet"
    print(f"  {row['Symbol']:12} "
          f"{row['Cap Category']:14} "
          f"{row['Consol Days']:5}d "
          f"{row['range_pct']:6.1f}% "
          f"{pct:+7.1f}%  "
          f"{wvol:.2f}x  "
          f"{bvol:.2f}x  "
          f"{bvol_tag:12}  [{status}]")

# ── BEARISH REVERSAL WATCH ────────────────────────────────────────────────────
print(f"\n{'─'*78}")
print(f"  ⚠  BEARISH REVERSAL WATCH")
print(f"  Stocks showing distribution signals — consider partial profit booking (25-30%)")
print(f"  Signal fires when 3 or more of 5 filters trigger")
print(f"{'─'*78}")
print(f"  Filters: (1)Runup>30%  (2)RSI>70×5d  (3)RSI+MACD divergence")
print(f"           (4)ADX fading  (5)Volume declining on up days")
print(f"{'─'*78}")

triggered_stocks = [(s, v) for s, v in bearish_results.items() if v['signal']]

if not triggered_stocks:
    print(f"\n  No stocks triggered this week. Market is clean.")
else:
    for symbol, result in triggered_stocks:
        score   = result['score']
        details = result['details']
        row     = tech_df[tech_df['Symbol'] == symbol].iloc[0]
        f1 = '✓' if details.get('filter1_runup')        else '✗'
        f2 = '✓' if details.get('filter2_rsi_extended') else '✗'
        f3 = '✓' if details.get('filter3_both_div')     else '✗'
        f4 = '✓' if details.get('filter4_adx_fade')     else '✗'
        f5 = '✓' if details.get('filter5_vol_decline')  else '✗'
        if   score == 5: inference = "Very strong distribution signal → book 25-30% immediately"
        elif score == 4: inference = "Strong signal → start scaling out 20-25%"
        else:            inference = "Early warning → monitor closely, avoid adding"
        print(f"""
  ⚠  {symbol}  ({row['Sector']})  Rs{row['Current Price']:.2f}  Score:{score}/5
     Runup from 52w low : {details.get('runup_pct', 0):+.1f}%
     Filters fired      : (1){f1} Runup  (2){f2} RSI>70×5d  (3){f3} Divergence  (4){f4} ADX fade  (5){f5} Vol↓
     RSI divergence     : {details.get('rsi_divergence')}  |  MACD divergence: {details.get('macd_divergence')}
     ADX peak→now       : {details.get('adx_peak', 0):.1f} → {details.get('adx_now', 0):.1f}  (fading trend strength)
     Sector             : {row['Sector Trend']} | {row['Sector Detail']}
     Action             : {inference}""")

print(f"\n{'─'*78}")
print(f"  Note: This is NOT a sell signal. Quality stocks may continue higher.")
print(f"  Use to protect partial gains. Re-enter if pullback holds key levels.")
print(f"{'─'*78}")

# ── FOOTER ────────────────────────────────────────────────────────────────────
print(f"\n{'─'*78}")
print(f"  L=Large Cap  ML=Mini Large Cap  M=Mid Cap  S=Small Cap")
print(f"  SecRank  = rank vs same sector | CapRank = rank vs same sector + cap")
print(f"  Week Vol = this week avg vs prior 15-day avg")
print(f"  Break Vol= this week avg vs consolidation period avg")
print(f"{'─'*78}")

# ── Save ──────────────────────────────────────────────────────────────────────
tech_df.to_csv('../data/technical_report.csv', index=False)
print("\nSaved: data/technical_report.csv ✅")

  AI STOCK SCREENER — WEEKLY REPORT
  21 March 2026

  SecRank  = rank vs same sector  |  CapRank = rank vs same sector + same cap
  Week Vol = this week avg vs prior 15-day avg
  Break Vol= this week avg vs consolidation period avg (breakout signal)

──────────────────────────────────────────────────────────────────────────────
  SECTOR TREND SUMMARY  (full market)
──────────────────────────────────────────────────────────────────────────────
  Sector                                     Trend                        Detail
  ────────────────────────────────────────────────────────────────────────────
  Agriculture                                Strong Downtrend ↓↓          RSI 34.4 | ADX 42.3 | MACD ▼ | EMA mixed | Below EMA200
  Automobile and Auto Components             Strong Downtrend ↓↓          RSI 37.2 | ADX 40.3 | MACD ▼ | EMA mixed | Below EMA200
  Banking                                    Strong Downtrend ↓↓          RSI 26.1 | ADX 53.8 | MACD ▼ | EMA mixed | Below EMA200
  